# Phase 10 — LLM Closed-Set Classification -> Topic-Driven Retrieval

LLM 에게 정확한 citation 생성을 안 시킴. 고정 18개 토픽 중 ID 만 고르게 함 (7B 강점=의미이해).
정확한 citation 은 BGE-M3 검색 + corpus 검증 담당 (환각 0).

플로우: 사건 -> seeds -> LLM 토픽 분류 -> 토픽별 독일어쿼리 multi-query 검색(laws+court)
-> 우선코드 부스트 -> corpus 검증 -> emit. 캐시 cache_phase3 재사용. GPU T4x2, Internet ON.
Add: bge-m3 + Mistral-7B-Instruct-v0.1


In [ ]:
# Cell 1 — env, paths, pip
!pip install -q rank_bm25 sentence-transformers faiss-cpu bitsandbytes accelerate

import re, csv, json, pickle, time, gc, sys, os
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

DATA = None
for cand in [
    Path('/kaggle/input/llm-agentic-legal-information-retrieval'),
    Path('/kaggle/input/competitions/llm-agentic-legal-information-retrieval'),
]:
    if cand.exists() and (cand / 'laws_de.csv').exists():
        DATA = cand; break
if DATA is None:
    raise RuntimeError(f'Data not found. /kaggle/input contents: {list(Path("/kaggle/input").iterdir())}')

WORK = Path('/kaggle/working')
CACHE = WORK / 'cache_phase3'
CACHE.mkdir(parents=True, exist_ok=True)
print(f'DATA: {DATA}')
print(f'WORK: {WORK}')

for p in ['train.csv', 'val.csv', 'test.csv', 'laws_de.csv', 'court_considerations.csv']:
    fp = DATA / p
    print(f'  {p:30s} {"ok" if fp.exists() else "MISSING"}  ({fp.stat().st_size/1e6:.1f} MB)' if fp.exists() else f'  {p:30s} MISSING')

# Toggles
USE_TRANSLATE = True       # NLLB EN→DE (both NLLB + LLM live; freest-GPU picker handles memory)
USE_RERANKER  = False      # phase 8: not used (LLM-first)
USE_COURT     = True       # phase 9: BGE-only court (cheap)       # court_considerations 2-stage (most expensive)
USE_LLM       = True       # phase 8 CORE — LLM-first generation; must be True. (orig note: set True only if you've attached the model + have time


In [ ]:
import json

# Cell 2 — Multilingual abbreviation map (FR/IT → DE) — 1662 entries from Omnilex starter repo
MULTILANG_ABBR = {'OIAA': 'AAMV', 'OMAA': 'HVUV', 'DE-OMBat': 'AB-VASm', 'OBE-FINMA': 'ABV-FINMA', 'OAdo': 'AdoV', 'OAdoz': 'AdoV', 'SDA': 'ADS', 'ORAT': 'AEFV', 'OIAgr': 'AEV', 'LMCFA': 'AFZFG', 'LMCCE': 'AFZFG', 'OMCFA': 'AFZFV', 'OMCCE': 'AFZFV', 'LAVS': 'AHVG', 'RAVS': 'AHVV', 'OAVS': 'AHVV', 'LEAR': 'AIAG', 'LSAI': 'AIAG', 'OEAR': 'AIAV', 'OSAIn': 'AIAV', 'LEI': 'AIG', 'LStrI': 'AIG', 'OAccD': 'AkkBV', 'AccredO-LPsy': 'AkkredV-PsyG', 'OEAc-LPPsi': 'AkkredV-PsyG', 'LEDPP': 'ALBAG', 'LSRPP': 'ALBAG', 'OEDPP': 'ALBAV', 'OSRPP': 'ALBAV', 'OBat': 'AlgV', 'OSRA': 'V-NDA', 'OdA': 'AlkBestV', 'OTAl': 'AlkBestV', 'LAlc': 'AlkG', 'OAlc': 'AlkV', 'OAllerg': 'AllergV', 'OGEmol': 'AllgGebV', 'OgeEm': 'AllgGebV', 'OSites': 'AltlV', 'OSiti': 'AltlV', 'OSI-AC': 'ALV-IsV', 'OAMéd': 'AMBV', 'OAMed': 'AMBV', 'OEMéd': 'AMZV', 'OOMed': 'AMZV', 'OOrgA': 'AO', 'OEs': 'AO', 'OOS': 'AOV', 'OOV': 'AOV', 'LTr': 'ArG', 'LL': 'ArG', 'OLT 1': 'ArGV 1', 'OLL 1': 'ArGV 1', 'OLT 2': 'ArGV 2', 'OLL 2': 'ArGV 2', 'OLT 3': 'ArGV 3', 'OLL 3': 'ArGV 3', 'OLT 4': 'ArGV 4', 'OLL 4': 'ArGV 4', 'OLT 5': 'ArGV 5', 'OLL 5': 'ArGV 5', 'OITRV': 'ARPV', 'OTR 1': 'ARV 1', 'OLR 1': 'ARV 1', 'OTR 2': 'ARV 2', 'OLR 2': 'ARV 2', 'LSEtr': 'ASG', 'LSEst': 'ASG', 'Limpauto': 'AStG', 'LIAut': 'AStG', 'Oimpauto': 'AStV', 'OIAut': 'AStV', 'OSur-ASR': 'ASV-RAB', 'OS-ASR': 'ASV-RAB', 'LAsi': 'AsylG', 'OA 1': 'AsylV 1', 'OAsi 1': 'AsylV 1', 'OA 2': 'AsylV 2', 'OAsi 2': 'AsylV 2', 'OA 3': 'AsylV 3', 'OAsi 3': 'AsylV 3', 'LTrAlp': 'AtraG', 'LTAlp': 'AtraG', 'Otransa': 'AtraV', 'OTrAl': 'AtraV', 'LPGA': 'ATSG', 'OPGA': 'ATSV', 'RSTF': 'AufRBGer', 'RVTF': 'AufRBGer', 'OAsc': 'AufzV', 'OSAC': 'AuLaV', 'OAEs': 'AuLaV', 'LECCT': 'AVEG', 'LOCCL': 'AVEG', 'OFAC': 'AVFV', 'OFAD': 'AVFV', 'LSE': 'AVG', 'LC': 'AVG', 'LACI': 'AVIG', 'LADI': 'AVIG', 'OACI': 'AVIV', 'OADI': 'AVIV', 'OS': 'AVO', 'OS-FINMA': 'AVO-FINMA', 'OSE': 'AVV', 'OC': 'AVV', 'LDI': 'AwG', 'LDT': 'AZG', 'LDL': 'AZG', 'OLDT': 'AZGV', 'OLDL': 'AZGV', 'ODMA': 'BAlV', 'LB': 'BankG', 'LBCR': 'BankG', 'OB': 'BankV', 'OBCR': 'BankV', 'OTConst': 'BauAV', 'OLCostr': 'BauAV', 'LPCo': 'BauPG', 'LProdC': 'BauPG', 'OPCo': 'BauPV', 'OProdC': 'BauPV', 'LFPr': 'BBG', 'OFPr': 'BBV', 'LTI': 'BEG', 'LTCo': 'BEG', 'LHand': 'BehiG', 'LDis': 'BehiG', 'OHand': 'BehiV', 'ODis': 'BehiV', 'ONo-ASR': 'BekV-RAB', 'OC-ASR': 'BekV-RAB', 'LStup': 'BetmG', 'OCStup': 'BetmKV', 'OEPStup': 'BetmPV', 'OSPStup': 'BetmPV', 'OAStup': 'BetmSV', 'ODStup': 'BetmSV', 'OTStup-DFI': 'BetmVV-EDI', 'OEStup-DFI': 'BetmVV-EDI', 'OProP': 'BevSV', 'OPPop': 'BevSV', 'LFAIE': 'BewG', 'LAFE': 'BewG', 'OAIE': 'BewV', 'OAFE': 'BewV', 'LF-CLaH': 'BG-HAÜ', 'LF-CAA': 'BG-HAÜ', 'LTEJUS': 'BG-RVUS', 'LTAGSU': 'BG-RVUS', 'LAr': 'BGA', 'LDFR': 'BGBB', 'LMI': 'BGBM', 'LCITES': 'BGCITES', 'LF-CITES': 'BGCITES', 'RTF': 'BGerR', 'LFSP': 'BGF', 'LLCA': 'BGFA', 'LTF': 'BGG', 'LDEA': 'BGIAA', 'LSISA': 'BGIAA', 'LBCF': 'BGLE', 'LRFF': 'BGLE', 'LPPS': 'BGMD', 'LDPS': 'BGMD', 'LFPC': 'BGMK', 'LTrans': 'BGÖ', 'LTras': 'BGÖ', 'LEAC': 'BGRB', 'LIAC': 'BGRB', 'LJAr': 'BGS', 'LGD': 'BGS', 'LTN': 'BGSA', 'LLN': 'BGSA', 'LOST': 'BGST', 'LFSI': 'BGST', 'LFIF': 'BIFG', 'OBiG': 'BiGV', 'OIB-FINMA': 'BIV-FINMA', 'LCESF': 'BiZG', 'LCSFS': 'BiZG', 'LCMIF': 'BIZMB', 'OMPr': 'BMV', 'LMP': 'BöB', 'LAPub': 'BöB', 'OPDC': 'BPDV', 'OPDPers': 'BPDV', 'LSIP': 'BPI', 'LDP': 'BPR', 'LPSP': 'BPS', 'RNC': 'BSO', 'LSF': 'BStatG', 'LStat': 'BStatG', 'LIB': 'BStG', 'RAATPF': 'BStGerNR', 'ROATPF': 'BStGerNR', 'ROTPF': 'BStGerOR', 'RFPPF': 'BStKR', 'RSPPF': 'BStKR', 'OBioc': 'VBP', 'OBcarb': 'BTrV', 'LN': 'BüG', 'LCit': 'BüG', 'LSCPT': 'BÜPF', 'OREE': 'BURV', 'ORIS': 'BURV', 'OLN': 'VOSA', 'OCit': 'BüV', 'Cst.': 'BV', 'Cost.': 'BV', 'LPP': 'BVG', 'OPP 1': 'BVV 1', 'OPP 2': 'BVV 2', 'OPP 3': 'BVV 3', 'LIDV': 'BVVG', 'LDDV': 'BVVG', 'LMSI': 'BWIS', 'PCF': 'BZP', 'PC': 'BZP', 'OCart': 'CartV', 'LChim': 'ChemG', 'LPChim': 'ChemG', 'OEChim': 'ChemGebV', 'OEPChim': 'ChemGebV', 'OPICChim': 'ChemPICV', 'ORRChim': 'ChemRRV', 'ORRPChim': 'ChemRRV', 'OChim': 'ChemV', 'OPChim': 'ChemV', 'OCPCh': 'ChKV', 'OCCC': 'ChKV', 'CRTIF': 'COTIF 1980', 'LCaS-COVID-19': 'Covid-19-SBüG', 'LFiS-COVID-19': 'Covid-19-SBüG', 'OCyS': 'CSV', 'OCS': 'GGBV', 'CAC': 'CWÜ', 'OACP': 'CZV', 'OAut': 'CZV', 'ACCEES': 'ASEEGKVBSPMSA', 'ACSCSSS': 'ASEEGKVBSPMSA', 'LIFD': 'DBG', 'OSRP': 'DBV', 'OTDD': 'DBZV', 'LDes': 'DesG', 'ODes': 'DesV', 'OSEP': 'DGV', 'OSAP': 'DGV', 'RSA': 'DRA', 'RSE': 'DRA', 'LPD': 'DSG', 'OEng': 'DüV', 'OCon': 'DüV', 'OD-ASR': 'DV-RAB', 'OPD': 'DZV', 'LCdF': 'EBG', 'Lferr': 'EBG', 'OCF': 'EBV', 'Oferr': 'EBV', 'OITE-PT': 'EDAV-DS', 'OITE-PT-DFI': 'EDAV-DS-EDI', 'OITE-UE': 'EDAV-EU', 'OITE-UE-DFI': 'EDAV-EU-EDI', 'OITE-AC': 'EDAV-Ht', 'OITEAc': 'EDAV-Ht', 'OEES': 'EESV', 'O-HEFSM': 'EHSM-V', 'O-SUFSM': 'EHSM-V', 'OEmV': 'EichGebV', 'OEm-V': 'EichGebV', 'OIPR': 'EigVV', 'RIPP': 'EigVV', 'LIFM': 'EIMG', 'OIFM': 'EIMV', 'OO': 'EiV', 'OU': 'EiV', 'OCCP': 'EKBV', 'OCSC': 'EKBV', 'OESC-OFAG': 'EKV-BLW', 'OVC-UFAG': 'EKV-BLW', 'LIE': 'EleG', 'LPC': 'ELG', 'OPC-AVS/AI': 'ELV', 'LMETA': 'EMBAG', 'LMeCA': 'EMBAG', 'OMETA': 'EMBAV', 'OMeCA': 'EMBAV', 'LEmb': 'EmbG', 'OESMB': 'EmGvV', 'OACMB': 'EmGvV', 'LCMP': 'EMKG', 'OCMP': 'EMKV', 'OIMepe': 'EMmV', 'OSMisE': 'EMmV', 'CSDLLF': 'KSMG', 'CSDDDLF': 'KSMG', 'OEEE': 'EnEV', 'OEEne': 'EnEV', 'OEneR': 'EnFV', 'OPEn': 'EnFV', 'LIFSN': 'ENSIG', 'OIFSN': 'ENSIV', 'LDét': 'EntsG', 'LDist': 'EntsG', 'Odét': 'EntsV', 'ODist': 'EntsV', 'OAAE': 'EÖBV', 'OAPuE': 'EÖBV', 'OAAE-DFJP': 'EÖBV-EJPD', 'OAPuE-DFGP': 'EÖBV-EJPD', 'LAPG': 'EOG', 'LIPG': 'EOG', 'OAPG': 'EOV', 'OIPG': 'EOV', 'OFDEP': 'EPDFV', 'OFCIP': 'EPDFV', 'LDEP': 'EPDG', 'LCIP': 'EPDG', 'ODEP': 'EPDV', 'OCIP': 'EPDV', 'ODEP-DFI': 'EPDV-EDI', 'OCIP-DFI': 'EPDV-EDI', 'LEp': 'EpG', 'CBE 2000': 'EPÜ 2000', 'OEp': 'EpV', 'OFR': 'ERV', 'OFoP': 'ERV', 'CE-TAF': 'ERV-BVGer', 'OUC': 'ESV', 'OIConf': 'ESV', 'Oexpa': 'ExpaV', 'Oespa': 'ExpaV', 'LAFam': 'FamZG', 'OAFam': 'FamZV', 'OAFami': 'FamZV', 'OEV 4': 'FAV 4', 'OEA 4': 'FAV 4', 'OSVo': 'FDO', 'LSFin': 'FIDLEG', 'LSerFi': 'FIDLEG', 'OSFin': 'FIDLEV', 'OSerFi': 'FIDLEV', 'LERI': 'FIFG', 'LPRI': 'FIFG', 'OECin': 'FiFV', 'OPCin': 'FiFV', 'LIMF': 'FinfraG', 'LInFi': 'FinfraG', 'OIMF': 'FinfraV', 'OInFi': 'FinfraV', 'OIMF-FINMA': 'FinfraV-FINMA', 'OInFi-FINMA': 'FinfraV-FINMA', 'LEFin': 'FINIG', 'LIsFi': 'FINIG', 'OEFin': 'FINIV', 'OIsFi': 'FINIV', 'OEFin-FINMA': 'FINIV-FINMA', 'OIsFi-FINMA': 'FINIV-FINMA', 'Oém-FINMA': 'FINMA-GebV', 'Oem-FINMA': 'FINMA-GebV', 'OA-FINMA': 'FINMA-PV', 'LFINMA': 'FINMAG', 'OMPRI': 'FIPBV', 'LFiEl': 'FiREG', 'LAiSE': 'FiREG', 'CRSR': 'Abkommen\nüber die Rechtsstellung der Flüchtlinge', 'CSSR': 'Abkommen\nüber die Rechtsstellung der Flüchtlinge', 'LCF': 'FKG', 'OReCI': 'FKINV', 'OPRCI': 'FKINV', 'LFA': 'FLG', 'LAF': 'FLG', 'RFA': 'FLV', 'OA Fam': 'FLV', 'OLALA': 'FMBV', 'OLAlA': 'FMBV', 'LPMA': 'FMedG', 'LPAM': 'FMedG', 'OPMA': 'VLIp', 'OMP': 'VöB', 'LTC': 'FMG', 'OSALA': 'FMV', 'OsAlA': 'FMV', 'OFOrg': 'FOrgV', 'OAOrg': 'FOrgV', 'OH': 'FPV', 'OOra': 'FPV', 'OQICin': 'FQIV', 'OQIC': 'FQIV', 'ODE': 'FrSV', 'OEDA': 'FrSV', 'LFus': 'FusG', 'OMCo': 'FV', 'OMaeC': 'FV', 'OF-SCPT': 'FV-ÜPF', 'AECSDPC': 'ASEEGMF', 'ACSCS': 'ASEEGMF', 'OAG': 'GaGV', 'OAppG': 'GaGV', 'AGTDC': 'GATT', 'ORF': 'GBV', 'REmol-TAF': 'GebR-BVGer', 'TA-TAF': 'GebR-BVGer', 'REmol-TFB': 'GebR-PatGer', 'TA-TFB': 'GebR-PatGer', 'Olico': 'GeBüV', 'Olc': 'GeBüV', 'OE ArchF': 'GebV BAR', 'OE ARF': 'GebV BAR', 'OEmol-AFC': 'GebV ESTV', 'OEm-AFC': 'GebV ESTV', 'OELP': 'GebV SchKG', 'OTLEF': 'GebV SchKG', 'Oem-LEI': 'GebV-AIG', 'OEmol-LStrI': 'GebV-AIG', 'Oem-Acc': 'GebV-Akk', 'Oemo-Acc': 'GebV-Akk', 'OEmol-LTr': 'GebV-ArG', 'OEm-LL': 'GebV-ArG', 'OEmol-OFRO': 'GebV-ASTRA', 'OEmo-USTRA': 'GebV-ASTRA', 'OEmol-LSE': 'GebV-AVG', 'OEm-LC': 'GebV-AVG', 'OEmol-OFEV': 'GebV-BAFU', 'OE-UFAM': 'GebV-BAFU', 'OEmol-OFSPO': 'GebV-BASPO', 'OEm UFSPO': 'GebV-BASPO', 'OEmol-OFAC': 'GebV-BAZL', 'OEm-UFAC': 'GebV-BAZL', 'Oem-OFJ': 'GebV-BJ', 'Oem UFG': 'GebV-BJ', 'OEmol-OFAG': 'GebV-BLW', 'Ordinanza sulle tasse UFAG': 'GebV-BLW', 'OEmol-DFAE': 'GebV-EDA', 'OEm-DFAE': 'GebV-EDA', 'OEmol-DFI-BN': 'GebV-EDI-NBib', 'OEm-DFI-BN': 'GebV-EDI-NBib', 'OEmol-CMP': 'GebV-EMK', 'OEm-CMP': 'GebV-EMK', 'Oémol-En': 'GebV-En', 'OE-En': 'GebV-En', 'OEmol-ASF': 'GebV-ESA', 'OEm-AVF': 'GebV-ESA', 'OEmol-fedpol': 'GebV-fedpol', 'OEm-fedpol': 'GebV-fedpol', 'OREDT': 'GebV-FMG', 'OTST': 'GebV-FMG', 'OEmol-RC': 'GebV-HReg', 'OTa-IPI': 'GebV-IGE', 'OEmol-LCart': 'GebV-KG', 'OEm-LCart': 'GebV-KG', 'OEm-METAS': 'GebV-METAS', 'Oem-METAS': 'GebV-METAS', 'OEmol-BN': 'GebV-NBib', 'OEm-BN': 'GebV-NBib', 'OEmol-TP': 'GebV-öV', 'OEm-TP': 'GebV-öV', 'OEmol-Publ': 'GebV-Publ', 'OEm-Pub': 'GebV-Publ', 'OÉmol-CSA': 'GebV-SAR', 'OEm-CSA': 'GebV-SAR', 'OEmol-SEFRI': 'GebV-SBFI', 'OEm-SEFRI': 'GebV-SBFI', 'OE-RaP': 'GebV-StS', 'OEm-RaP': 'GebV-StS', 'OEmol-OHB': 'GebV-TPS', 'OEmSON': 'GebV-TPS', 'OEmol-DDPS': 'GebV-VBS', 'OEm-DDPS': 'GebV-VBS', 'OÉIPPSS': 'GebVO St', 'OSEIPSS': 'GebVO St', 'LGéo': 'GeoIG', 'LGI': 'GeoIG', 'OGéo': 'GeoIV', 'OGI': 'GeoIV', 'OGéo-swisstopo': 'GeoIV-swisstopo', 'OGI-swisstopo': 'GeoIV-swisstopo', 'OGéom': 'GeomV', 'Ogeom': 'GeomV', 'ONGéo': 'GeoNV', 'ONGeo': 'GeoNV', 'ORPSan': 'GesBAV', 'LPSan': 'GesBG', 'OCPSan': 'GesBKV', 'OSAS': 'GGBV', 'OCMD': 'GGUV', 'OMCont': 'GGUV', 'OIC-DFI': 'GgV-EDI', 'LCB': 'PAG', 'LBDI': 'GKG', 'ODVo': 'GKZ', 'OCPo': 'GKZ', 'LEg': 'GlG', 'LPar': 'GlG', 'OBPL': 'GLPV', 'RTFB': 'GR-PatGer', 'RI-COMCO': 'GR-WEKO', 'RCN': 'GRN', 'RCE': 'GRS', 'RCS': 'GRS', 'LEaux': 'GSchG', 'LPAc': 'GSchG', 'OEaux': 'GSchV', 'OPAc': 'GSchV', 'LGG': 'GTG', 'LIG': 'GTG', 'LAGH': 'GUMG', 'LEGU': 'GUMG', 'OAGH': 'GUMV', 'OEGU': 'GUMV', 'LTM': 'GüTG', 'OTM': 'GüTV', 'LTTM': 'GVVG', 'LTrasf': 'GVVG', 'LBA': 'GwG', 'LRD': 'GwG', 'OBA': 'GwV', 'ORD': 'GwV', 'OBA-OFDF': 'GwV-BAZG', 'ORD-UDSC': 'GwV-BAZG', 'OBA-DFJP': 'GwV-EJPD', 'ORD-DFGP': 'GwV-EJPD', 'OBA-CFMJ': 'GwV-ESBK', 'ORD-CFCG': 'GwV-ESBK', 'OBA-FINMA': 'GwV-FINMA', 'ORD-FINMA': 'GwV-FINMA', 'LTrD': 'HArG', 'LLD': 'HArG', 'OTrD': 'HArGV', 'OLLD': 'HArGV', 'OIPSD': 'HasLV', 'OIPSDA': 'HasLV', 'OIPSD-DEFR': 'HasLV-WBF', 'OIPSDA-DEFR': 'HasLV-WBF', 'OPFP-FINMA': 'HBEV-FINMA', 'LRH': 'HFG', 'LRUm': 'HFG', 'LEHE': 'HFKG', 'LPSU': 'HFKG', 'OMCR 20': 'HFMV 20', 'OPCR 20': 'HFMV 20', 'OMCR 22': 'HFMV 22', 'OPCR 22': 'HFMV 22', 'ORH': 'HFV', 'ORUm': 'HFV', 'LLGV': 'HGVAnG', 'LRAV': 'HGVAnG', 'OCBo': 'HHV', 'OCoL': 'HHV', 'CLaH96': 'HKsÜ', 'Convenzione dell’Aia sulla protezione dei minori': 'HKsÜ', 'OGOM': 'HKSV', 'OGOE': 'HKSV', 'LPTh': 'HMG', 'LATer': 'HMG', 'ORC': 'HRegV', 'OCCHE': 'HSBBV', 'OSCU': 'HSBBV', 'OMAV': 'HVA', 'OMAI': 'HVI', 'OMAINF': 'HVUV', 'OHyg': 'HyV', 'ORI': 'HyV', 'OIAM': 'IAMV', 'OSFPrHE': 'IBH-V', 'O-SIFPU': 'IBH-V', 'LSIS': 'SIaG', 'LSISpo': 'IBSG', 'OSIS': 'IBSV', 'OSISpo': 'IBSV', 'OMCC': 'VSFK', 'OCoCr': 'IBTV', 'OId-BDTA': 'IdTVD-V', 'OIBDTA': 'IdTVD-V', 'LIPPI': 'IFEG', 'LIPIn': 'IFEG', 'OIPI': 'IGE-OV', 'Oper-IPI': 'IGE-PersV', 'OPer-IPI': 'IGE-PersV', 'LIPI': 'IGEG', 'OAiR': 'InkHV', 'OAInc': 'InkHV', 'Oinv': 'InvV', 'OPICin': 'IPFiV', 'LDIP': 'IPRG', 'LISint': 'IQG', 'LIFI': 'IQG', 'RInfo-TFB': 'IR-PatGer', 'EIMP': 'IRSG', 'AIMP': 'IRSG', 'OEIMP': 'IRSV', 'OAIMP': 'IRSV', 'O-SI ABV': 'ISABV-V', 'O-SIAMV': 'ISABV-V', 'LSI': 'ISG', 'LSIn': 'ISG', 'O-SICAL': 'ISLK-V', 'O-SIFA': 'ISLK-V', 'OSIAgr': 'ISLV', 'OSIP-OFDF': 'IStrV-BAZG', 'OSIP-UDSC': 'IStrV-BAZG', 'OSAR': 'ISUV', 'OSIStr': 'ISUV', 'ODiv': 'IvDV', 'ODIV': 'IvDV', 'LAI': 'IVG', 'RAI': 'IVV', 'OAI': 'IVV', 'OSIAC': 'IVZV', 'O OFSPO J+S': 'J+S-V-BASPO', 'O-G+S-UFSPO': 'J+S-V-BASPO', 'LPMFJ': 'JSFVG', 'LPMFV': 'JSFVG', 'OPMFJ': 'JSFVV', 'OPMFV': 'JSFVV', 'LChP': 'JSG', 'LCP': 'JSG', 'DPMin': 'JStG', 'PPMin': 'JStPO', 'OChP': 'JSV', 'OCP': 'VKL', 'OFPC-FINMA': 'KAKV-FINMA', 'OFICol-FINMA': 'KAKV-FINMA', 'OAAcc': 'KBFHV', 'OACust': 'KBFHV', 'LENu': 'KEG', 'OENu': 'KEV', 'LEC': 'KFG', 'LPCu': 'KFG', 'OAVAuto': 'KFZV', 'OAuto': 'KFZV', 'LTBC': 'KGTG', 'OTBC': 'KGTV', 'OIBC': 'KGVV', 'OEBC': 'KGVV', 'LRCN': 'KHG', 'ORCN': 'KHV', 'LIC': 'KIG', 'LEEJ': 'KJFG', 'LPAG': 'KJFG', 'OEEJ': 'KJFV', 'OPAG': 'KJFV', 'LCC': 'KKG', 'OPCC': 'KKV', 'OICol': 'KKV', 'OPC-FINMA': 'KKV-FINMA', 'OICol-FINMA': 'KKV-FINMA', 'OClin': 'KlinV', 'OSRUm': 'KlinV', 'OClin-Dim': 'KlinV-Mep', 'OSRUm-Dmed': 'KlinV-Mep', 'OCI': 'KlV', 'OOCli': 'KlV', 'LFMG': 'KMG', 'LMB': 'KMG', 'OMG': 'KMV', 'OMB': 'KMV', 'OAOF': 'KOV', 'RUF': 'KOV', 'OCoo': 'KoVo', 'OCCRT': 'KoVo', 'OAMédcophy': 'KPAV', 'OMCF': 'KPAV', 'OCPF': 'KPFV', 'FP-TFB': 'KR-PatGer', 'SpG-TFB': 'KR-PatGer', 'OCré-FINMA': 'KreV-FINMA', 'OCre-FINMA': 'KreV-FINMA', 'OEMO': 'KRV', 'ORMT': 'KRV', 'Cst-GE': 'KV-GE', 'Cost.-GE': 'KV-GE', 'LSAMal': 'KVAG', 'LVAMal': 'KVAG', 'LAMal': 'KVG', 'OAMal': 'KVV', 'OPVA': 'LAfV', 'OPSAgr': 'LAfV', 'LRA': 'LBG', 'OAgrD': 'LDV', 'ODAgr': 'LDV', 'OLEl': 'LeV', 'LA': 'LFG', 'LNA': 'LFG', 'OSAv': 'LFV', 'ONA': 'LFV', 'OIBL': 'LGBV', 'OULiq': 'LGBV', 'OGN': 'LGeoIV', 'ODAlOUs': 'LGV', 'ODerr': 'LGV', 'OLiq': 'LiqV', 'OIDAl': 'LIV', 'OID': 'LIV', 'LDAl': 'LMG', 'LDerr': 'LMG', 'OIMLo': 'LMmV', 'OSML': 'LMmV', 'OELDAl': 'LMVV', 'OELDerr': 'LMVV', 'LBFA': 'LPG', 'LAAgr': 'LPG', 'OLRO-FINMA': 'LROV-FINMA', 'OPair': 'LRV', 'OIAt': 'LRV', 'LPPM': 'LSMG', 'OPPM': 'LSMV', 'OPB': 'LSV', 'OIF': 'LSV', 'OTrA': 'LTrV', 'CL': 'LugÜ', 'CLug': 'LugÜ', 'LAP': 'LVG', 'OMN': 'LVV', 'OMN-DDPS': 'LVV-VBS', 'LAgr': 'LwG', 'OAcCP': 'MAkkV', 'OAGio': 'MAkkV', 'OBMa': 'MaLV', 'ORMAp': 'MaLV', 'OMar-FINMA': 'MarV-FINMA', 'OMer-FINMA': 'MarV-FINMA', 'OMach': 'MaschV', 'OMacch': 'MaschV', 'OMat': 'MatV', 'OMAT-DDPS': 'MatV', 'ORM': 'MAV', 'OPart': 'MBV', 'OParC': 'MBV', 'OCMil': 'MCAV', 'OCDM': 'MCAV', 'ODqua': 'MeAV', 'OIQ': 'MeAV', 'ODqua-DFJP': 'MeAV-EJPD', 'OIQ-DFGP': 'MeAV-EJPD', 'LPMéd': 'MedBG', 'LPMed': 'MedBG', 'OPMéd': 'MedBV', 'OPMed': 'MedBV', 'ODim': 'MepV', 'ODmed': 'MepV', 'OSRM': 'MeQV', 'LMétr': 'MessG', 'LMetr': 'MessG', 'OIMes': 'MessMV', 'OStrM': 'MessMV', 'LMét': 'MetG', 'LMet': 'MetG', 'OMét': 'MetV', 'OMet': 'MetV', 'OSV': 'MFV', 'OSVM': 'MFV', 'LAAM': 'MG', 'LM': 'MG', 'OBCBA': 'MGwV', 'OURD': 'MGwV', 'LSIA': 'MIG', 'LSIM': 'MIG', 'OIMin': 'MindStV', 'OImM': 'MindStV', 'OMinTA': 'MinLV', 'Limpmin': 'MinöStG', 'LIOm': 'MinöStG', 'Oimpmin': 'MinöStV', 'OIOm': 'MinöStV', 'LUMin': 'MinVG', 'OUMin': 'MinVV', 'OCL': 'MiPV', 'OSIAr': 'MIV', 'OSIM': 'MIV', 'OCM ES': 'MiVo-HF', 'OERic-SSS': 'MiVo-HF', 'OJM': 'MJV', 'O-GM': 'MJV', 'OAMil': 'MLFV', 'OPCNP': 'MNKPV', 'LPM': 'MSchG', 'OPM': 'MSchV', 'CPM': 'MStG', 'PPM': 'MStP', 'OJPM': 'MStV', 'OGPM': 'MStV', 'O sur la monnaie': 'MünzV', 'OMon': 'MünzV', 'LAM': 'MVG', 'OAM': 'MVV', 'LTVA': 'MWSTG', 'LIVA': 'MWSTG', 'OTVA': 'MWSTV', 'OIVA': 'MWSTV', 'LFORTA': 'NAFG', 'LFOSTRA': 'NAFG', 'ONag': 'NagV', 'OANS': 'NARV', 'OPANS': 'NARV', 'LBN': 'NBG', 'LBNS': 'NBibG', 'OBNS': 'NBibV', 'LRens': 'NDG', 'LAIn': 'NDG', 'ORens': 'NDV', 'OAIn': 'NDV', 'OMBT': 'NEV', 'OPBT': 'NEV', 'OPU': 'NFSV', 'OPE': 'PAVO', 'LPN': 'NHG', 'OPN': 'NHV', 'LRNIS': 'NISSG', 'ORNI': 'NISV', 'OIBT': 'NIV', 'OPCN-OCDE': 'NKPV-OECD', 'OPCN-OCSE': 'NKPV-OECD', 'LVA': 'NSAG', 'LUSN': 'NSAG', 'OVA': 'NSAV', 'OUSN': 'NSAV', 'LRN': 'NSG', 'LSN': 'NSG', 'ORN': 'NSV', 'OSN': 'NSV', 'OARF': 'NZV', 'OARF-OFT': 'NZV-BAV', 'OARF-UFT': 'NZV-BAV', 'OHS-LP': 'OAV-SchKG', 'OAV-LEF': 'OAV-SchKG', 'LAO': 'OBG', 'LMD': 'OBG', 'OAO': 'VUB', 'OMD': 'OBV', 'OPub-FINMA': 'OffV-FINMA', 'LAVI': 'OHG', 'LAV': 'OHG', 'OAVI': 'OHV', 'CO': 'OR', 'OCRDP': 'ÖREBKV', 'OCRDPP': 'ÖREBKV', 'OdelO': 'OrFV', 'OTOr': 'OrFV', 'Org-OMP': 'Org-VöB', 'OOAPub': 'Org-VöB', 'Org ChF': 'OV-BK', 'OrgCaF': 'OV-BK', 'Org CF': 'OV-BR', 'OOrg-CF': 'OV-BR', 'Org DFAE': 'OV-EDA', 'OOrg-DFAE': 'OV-EDA', 'Org DFI': 'OV-EDI', 'OOrg-DFI': 'OV-EDI', 'Org DFF': 'OV-EFD', 'Org-DFF': 'OV-EFD', 'Org DFJP': 'OV-EJPD', 'Org-DFGP': 'OV-EJPD', 'Org LRH': 'OV-HFG', 'Org-LRUm': 'OV-HFG', 'Org DETEC': 'OV-UVEK', 'Org-DATEC': 'OV-UVEK', 'Org-DDPS': 'OV-VBS', 'OOrg-DDPS': 'OV-VBS', 'Org DEFR': 'OV-WBF', 'Org-DEFR': 'OV-WBF', 'LCBr': 'PAG', 'LParl': 'ParlG', 'OLPA': 'VABFP', 'Oparl': 'ParlVV', 'LPart': 'PartG', 'LUD': 'PartG', 'OPTP': 'PaRV', 'OPFP': 'PaRV', 'LBI': 'PatG', 'LTFB': 'PatGG', 'OParcs': 'PäV', 'OPar': 'PäV', 'OAMin': 'PAVO', 'OPTA': 'PAVV', 'LTV': 'PBG', 'OPD-EPF': 'PDV-ETH', 'OPDP-PF': 'PDV-ETH', 'LLG': 'PfG', 'LOF': 'PfG', 'OLG': 'PfV', 'OOF': 'PfV', 'OSaVé': 'PGesV', 'OSalV': 'PGesV', 'OSaVé-DEFR-DETEC': 'PGesV-WBF-UVEK', 'OSalV-DEFR-DATEC': 'PGesV-WBF-UVEK', 'ORPGAA': 'PGRELV', 'ORFGAA': 'PGRELV', 'OPha': 'PhaV', 'OFarm': 'PhaV', 'LOP': 'POG', 'LRFP': 'PrHG', 'LRDP': 'PrHG', 'LSPro': 'PrSG', 'OSPro': 'PrSV', 'ORRTP': 'PRTR-V', 'OPRTR': 'PRTR-V', 'OEPI': 'PSAV', 'ODPI': 'PSAV', 'OPPh': 'PSMV', 'OPF': 'VSB', 'OPsy': 'PsyBV', 'OPPsi': 'PsyBV', 'LPsy': 'PsyG', 'LPPsi': 'PsyG', 'LSPr': 'PüG', 'OPers-METAS': 'PV-METAS', 'OPersTF': 'PVBger', 'OPers-PDHH': 'PVFMH', 'OPers-PRA': 'PVFMH', 'OPers-PDHH-DDPS': 'PVFMH-VBS', 'OPers-PRA-DDPS': 'PVFMH-VBS', 'OPersT': 'PVGer', 'OPers-EPF': 'PVO-ETH', 'OPers PF': 'PVO-ETH', 'OPers-ServAS': 'PVO-TVS', 'OPers-SAT': 'PVO-TVS', 'OPers-PPOE': 'PVSPA', 'OPers-POE': 'PVSPA', 'OPers-PPOE-DDPS': 'PVSPA-VBS', 'OPers-POE-DDPS': 'PVSPA-VBS', 'OIS': 'QStV', 'OIFo': 'QStV', 'OQuaDu': 'QuNaV', 'OQuSo': 'QuNaV', 'R-COPA': 'R-UEK', 'LSR': 'RAG', 'OSRev': 'RAV', 'OEPC-FINMA': 'RelV-FINMA', 'OAPC-FINMA': 'RelV-FINMA', 'RCETF': 'ReRBGer', 'ORe-DFI': 'ResV-EDI', 'ORAMal-DFI': 'ResV-EDI', 'LHR': 'RHG', 'LArRa': 'RHG', 'OHR': 'RHV', 'OArRa': 'RHV', 'OSITC': 'RLSV', 'OITC': 'RLV', 'OrX': 'RöV', 'LAT': 'RPG', 'LPT': 'RPG', 'OAT': 'RPV', 'OPT': 'RPV', 'LRTV': 'RTVG', 'ORTV': 'RTVV', 'OR-AVS': 'RV-AHV', 'LOGA': 'RVOG', 'OLOGA': 'RVOV', 'LASEI': 'SAFIG', 'LASPI': 'SAFIG', 'OPTM': 'SAMV', 'OPLM': 'SAMV', 'OAGa': 'SaV', 'OASal': 'SaV', 'LCFF': 'SBBG', 'LFFS': 'SBBG', 'OMAS': 'SBMV', 'OMSC': 'SBMV', 'LP': 'SchKG', 'LEF': 'SchKG', 'LICa': 'SebG', 'LIFT': 'SebG', 'OICa': 'SebV', 'OIFT': 'SebV', 'OFDG': 'SEFV', 'OFDS': 'SEFV', 'OCâbles': 'SeilV', 'OFuni': 'SeilV', 'OASRE': 'SERV-V', 'OARE': 'SERV-V', 'LASRE': 'SERVG', 'LARE': 'SERVG', 'OPAAb': 'SGV', 'OPeM': 'SGV', 'LEIS': 'SIaG', 'LISDC': 'SIRG', 'CRUGBIN': 'BASEVKGNMD', 'ACSRUGBIN': 'BASEVKGNMD', 'ORIn': 'SnAV', 'ORim': 'SnAV', 'OMJ-DFJP': 'SPBV-EJPD', 'OCG-DFGP': 'SPBV-EJPD', 'OSLing': 'SpDV', 'LLC': 'SpG', 'LLing': 'SpG', 'LESp': 'SpoFöG', 'LPSpo': 'SpoFöG', 'OESp': 'SpoFöV', 'OPSpo': 'SpoFöV', 'LExpl': 'SprstG', 'LEspl': 'SprstG', 'OExpl': 'SprstV', 'OEspl': 'SprstV', 'OLang': 'SpV', 'OLing': 'SpV', 'LVP': 'SRVG', 'LPV': 'SRVG', 'LESE': 'SSchG', 'LSSE': 'SSchG', 'OESE': 'SSchV', 'OSSE': 'SSchV', 'OESE-DFI': 'SSchV-EDI', 'OSSE-DFI': 'SSchV-EDI', 'LNM': 'SSG', 'OSR': 'SSV', 'OSStr': 'SSV', 'LECF': 'StADG', 'LOA': 'StAG', 'LImA': 'StAG', 'LAAF': 'StAhiG', 'OAAF': 'StAhiV', 'OSOA': 'StAV', 'OImA': 'StAV', 'LOAP': 'StBOG', 'OASF': 'STEBV', 'LRCS': 'StFG', 'LCel': 'StFG', 'OPAM': 'StFV', 'OPIR': 'StFV', 'CP': 'StGB', 'LHID': 'StHG', 'LAID': 'StHG', 'OIMRI': 'StMmV', 'OSMRI': 'StMmV', 'CPP': 'StPO', 'LCJ': 'StReG', 'LCaGi': 'StReG', 'OCJ': 'StReV', 'OCaGi': 'StReV', 'LApEl': 'StromVG', 'LAEl': 'StromVG', 'OApEl': 'StromVV', 'OAEl': 'StromVV', 'LRaP': 'StSG', 'ORaP': 'StSV', 'LEnTR': 'STUG', 'LPTS': 'STUG', 'OEnTR': 'STUV', 'OTStr': 'STUV', 'LTRA': 'STVG', 'LTS': 'STVG', 'LSu': 'SuG', 'LRPL': 'SVAG', 'LTTP': 'SVAG', 'ORPL': 'SVAV', 'OTTP': 'SVAV', 'LCR': 'SVG', 'LCStr': 'SVG', 'OS LCart': 'SVKG', 'LPTab': 'TabPG', 'OPTab': 'TabPV', 'OETV 1': 'TAFV 1', 'OETV 2': 'TAFV 2', 'OETV 3': 'TAFV 3', 'OMédv': 'TAMV', 'OMvet': 'TAMV', 'OPBD': 'TBDV', 'OPPD': 'TBDV', 'LVPC': 'TEVG', 'LRVC': 'TEVG', 'OTRF': 'TGBV', 'OSSAn': 'TGDV', 'LETC': 'THG', 'LOTC': 'THG', 'OIMTh': 'TMmV', 'OSMT': 'TMmV', 'LTo': 'ToG', 'OTo': 'ToV', 'OFPT': 'TPFV', 'LTro': 'TrG', 'LIF': 'TrG', 'OFPAn': 'TSchAV', 'LPA': 'TSchG', 'LPAn': 'TSchG', 'OPAn': 'TSchV', 'LFE': 'TSG', 'LTab': 'TStG', 'LImT': 'TStG', 'OITab': 'TStV', 'OImT': 'TStV', 'LET': 'TUG', 'LATC': 'TUG', 'OServAS': 'TVSV', 'OSAT': 'TVSV', 'OPPPS': 'TwwV', 'OPPS': 'VMD', 'LACRE': 'UEG', 'LSgrI': 'UEG', 'OOPA': 'UEV', 'O-COPA': 'UEV', 'Règlement sur la pêche dans le lac Inférieur': 'Unterseefischereiordnung', 'Ordinamento della pesca nel Lago Inferiore': 'Unterseefischereiordnung', 'LTSM': 'UGüTG', 'LTMS': 'UGüTG', 'LIDE': 'UIDG', 'LIDI': 'UIDG', 'OIDE': 'UIDV', 'OIDI': 'UIDV', 'LPtra': 'ÜLG', 'LPTD': 'ÜLG', 'OPtra': 'ÜLV', 'OPTD': 'ÜLV', 'OUMR': 'UraM', 'MMRa': 'UraM', 'LDA': 'URG', 'ODAu': 'URV', 'LPE': 'USG', 'LPAmb': 'USG', 'LAA': 'UVG', 'LAINF': 'UVG', 'OEIE': 'UVPV', 'OEIA': 'UVPV', 'OLAA': 'UVV', 'OAINF': 'UVV', 'LCD': 'UWG', 'LCSl': 'UWG', 'OPersMil': 'V Mil Pers', 'OPers mil': 'V Mil Pers', 'OSEtr': 'V-ASG', 'OSEst': 'V-ASG', 'O-LERI': 'V-FIFG', 'O-LPRI': 'V-FIFG', 'O-LERI-DEFR': 'V-FIFG-WBF', 'O-LPRI-DEFR': 'V-FIFG-WBF', 'OLEH': 'V-GSG', 'OSOsp': 'V-GSG', 'O-LEHE': 'V-HFKG', 'O-LPSU': 'V-HFKG', 'O-STAC': 'V-LTDB', 'O-SIEs': 'V-NDA', 'O-LRNIS': 'V-NISSG', 'O-CNC-FPr': 'V-NQR-BB', 'O QNQ FP': 'V-NQR-BB', 'OCommcon-EPF': 'V-Schliko-ETH', 'O CommConc PF': 'V-Schliko-ETH', 'O-AQIS': 'V-SQWI', 'O-GQIS': 'V-SQWI', 'O-CP-CPM': 'V-StGB-MSt', 'OCP-CPM': 'V-StGB-MSt', 'OPNA': 'VABFP', 'OCDoc': 'VABK', 'RCDoc': 'VABK', 'OETHand': 'VAböV', 'ORTDis': 'VAböV', 'OISofCA': 'VABUA', 'OFSPE': 'VABUA', 'OCFH': 'VAEW', 'OIFI': 'VAEW', 'LSA': 'VAG', 'OMHSI': 'VAI', 'OMFSI': 'VAI', 'OIFC': 'VAK', 'OCFQE': 'VAK', 'OMéd': 'VAM', 'OM': 'VAM', 'OIMEC': 'VAMF', 'OMGC': 'VAMF', 'OMSVM': 'VAmFD', 'OIGE': 'VAMV', 'OSGS': 'VAMV', 'Odac': 'VAN', 'OEAC': 'VAN', 'OSRens': 'VAND', 'OVAIn': 'VAND', 'OLPS': 'VAPF', 'OQPN': 'VAPK', 'OEPIN': 'VAPK', 'OTAS': 'VASA', 'OTaRSi': 'VASA', 'OMBat': 'VASm', 'ONCR': 'VASR', 'OITEP': 'VATPE', 'OITIP': 'VATPE', 'OAHST': 'VATT', 'OATFS': 'VATT', 'OAAFM': 'VATV', 'OASAM': 'VATV', 'OAAFM-DDPS': 'VATV-VBS', 'OASAM-DDPS': 'VATV-VBS', 'ODPO': 'VAU', 'OAPO': 'VAU', 'OInstr pré': 'VAusb', 'OISP': 'VAusb', 'OInstr prém DDPS': 'VAusb-VBS', 'OISP-DDPS': 'VAusb-VBS', 'OMO': 'VAV', 'OMU': 'VAV', 'OMO-DDPS': 'VAV-VBS', 'OMU-DDPS': 'VAV-VBS', 'OLDI': 'VAwG', 'ODI': 'VID', 'OASMéd': 'VAZV', 'OOSM': 'VAZV', 'ODFR': 'VBB', 'Osol': 'VBBo', 'O suolo': 'VBBo', 'OLAr': 'VBGA', 'OLFP': 'VBGF', 'OTrans': 'VBGÖ', 'OTras': 'VBGÖ', 'OIFP': 'VBLN', 'OTUIC': 'VBNIB', 'ODO': 'VBO', 'OOC-SCPT': 'VBO-ÜPF', 'OThand': 'VböV', 'OTDis': 'VböV', 'OPBio': 'VBP', 'OIOP': 'VBPO', 'OOCOP': 'VBPO', 'O-OPers': 'VBPV', 'O-OPers-DFAE': 'VBPV-EDA', 'ORE I': 'VBR I', 'ONE I': 'VBR I', 'ORCSN': 'VBRK', 'ORTN': 'VBRK', 'OEMFP': 'VBSTB', 'OSMFP': 'VBSTB', 'OPSEnt': 'VBSV', 'OPSAz': 'VBSV', 'OAdma': 'VBVA', 'OAE-AF': 'VBVA', 'OGPCT': 'VBVV', 'OABCT': 'VBVV', 'OESN': 'VBWK', 'OCGIN': 'VBWK', 'OCITES': 'VCITES', 'O-CITES': 'VCITES', 'OME-SCPT': 'VD-ÜPF', 'OE-SCPT': 'VD-ÜPF', 'OODA': 'VDA', 'OODE': 'VDA', 'OTPSP': 'VDPS', 'ODPSP': 'VDPS', 'OEDRP-DFI': 'VDPV-EDI', 'OSDRP-DFI': 'VDPV-EDI', 'OCPD': 'VDSZ', 'OTNI': 'VDTI', 'OTDI': 'VDTI', 'OACA': 'VDZV', 'ODCA': 'VDZV', 'OLQE': 'VEAF', 'OLAE': 'VEAF', 'OIELFP': 'VEAGOG', 'OIEVFF': 'VEAGOG', 'OEFMP': 'VEBMP', 'OEE-VVT': 'VEE-PLS', 'OEE-AAT': 'VEE-PLS', 'ODF': 'VEJ', 'OBAF': 'VEJ', 'OGE': 'VEKF', 'OCGE': 'VEKF', 'OEmiA': 'VEL', 'OVotE': 'VEleS', 'OVE': 'VEleS', 'OMETr': 'VEMZ', 'OSTAC': 'VEMZ', 'OESS': 'VES', 'OIIS': 'VES', 'OCREPF': 'VETHBK', 'OCRPF': 'VETHBK', 'OCEl-PA': 'VeÜ-VwV', 'OCE-PA': 'VeÜ-VwV', 'OCEl-PCPP': 'VeÜ-ZSSV', 'OCE-PCPE': 'VeÜ-ZSSV', 'OEV': 'VEV', 'OMoD': 'VeVA', 'OTRif': 'VeVA', 'O E-VERA': 'VEVERA', 'Ordinanza E-VERA': 'VEVERA', 'OIMA': 'VLIb', 'OIMeA': 'VFAI', 'OAFA': 'VFAL', 'OOIT': 'VFAV', 'OPer-Fu': 'VFB-B', 'OLAFum': 'VFB-B', 'OPer-D': 'VFB-DB', 'OADAP': 'VFB-DB', 'OPer-H': 'VFB-G', 'OAS-O': 'VFB-G', 'OPer-B': 'VFB-H', 'OASL': 'VFB-H', 'OPer-Fl': 'VFB-K', 'OASPR': 'VFB-K', 'OPer-AH': 'VFB-LG', 'OASAOG': 'VFB-LG', 'OPer-P': 'VFB-S', 'OALPar': 'VFB-S', 'OPer-S': 'VFB-SB', 'OASSP': 'VFB-SB', 'OPer-Fo': 'VFB-W', 'OASEF': 'VFB-W', 'OVCC': 'VFBF', 'OMA': 'VFD', 'OILAE': 'VFlaW', 'OLDDA': 'VFlaW', 'OLCP': 'VFP', 'Oform': 'VFRR', 'Rform': 'VFRR', 'OSNA': 'VFSD', 'OSA': 'VFSD', 'OAF': 'VFV', 'LRCF': 'VG', 'LResp': 'VG', 'OFCoop': 'VGeK', 'RFCoop': 'VGeK', 'LTAF': 'VGG', 'FITAF': 'VGKE', 'TS-TAF': 'VGKE', 'RTAF': 'VGR', 'OJAr': 'VGS', 'OGD': 'VGS', 'OSPEX': 'VGSEB', 'OASAE': 'VGSEB', 'OEB': 'VGV', 'OIB': 'VGV', 'ODAlGM': 'VGVL', 'ODerrGM': 'VGVL', 'ORegBL': 'VGWR', 'OREA': 'VREG', 'OGOC': 'VHBT', 'OGOCC': 'VHBT', 'OCont': 'VHK', 'OHyPL': 'VHyMP', 'OIgPL': 'VHyMP', 'OHyPPr': 'VHyPrP', 'OIPPrim': 'VHyPrP', 'OHyAb': 'VHyS', 'OIgM': 'VHyS', 'ODIn': 'VID', 'OSIA': 'VIL', 'OILC': 'VILB', 'OSIC': 'VIM', 'OICoM': 'VIM', 'OCMI': 'VIMK', 'OIE': 'VIntA', 'OIntS': 'VIntA', 'OPPEtr': 'VIPaV', 'OIPPE': 'VIPaV', 'OSIS-SRC': 'VIS-NDB', 'OSIME-SIC': 'VIS-NDB', 'OISOS': 'VISOS', 'OVIS': 'VISV', 'OITPTh': 'VITH', 'OITAT': 'VITH', 'OIVS': 'VIVS', 'OCISF': 'ViZG', 'OCISC': 'ViZG', 'OCMIF': 'VIZMB', 'OJAR-FSTD': 'VJAR-FSTD', 'OACata': 'VKA', 'OLCC': 'VKKG', 'OCCEA': 'VKKL', 'OCoC': 'VKKL', 'OSPBC': 'VKKP', 'OSBC': 'VKKP', 'OCPre': 'VKL', 'OCSN': 'VKNS', 'OCos': 'VKos', 'OCTSE': 'VKOVA', 'OPFr': 'VKP', 'OCPPME': 'VKP-KMU', 'OCPPMI': 'VKP-KMU', 'OSSC': 'VKSD', 'Ocach': 'VKSWk', 'OCRCI': 'VKSWk', 'OFA-FINMA': 'VKV-FINMA', 'OCTrI': 'VKVöV', 'OCTrIn': 'VKVöV', 'OMDA': 'VKZ', 'OCA': 'VVK', 'OBNP': 'VLBE', 'ODPPE': 'VLBE', 'OBCF': 'VLE', 'ORFF': 'VLE', 'ORAgr': 'VLF', 'LCo': 'VlG', 'OECA': 'VLHb', 'OICA': 'VLHb', 'OOMA': 'VLIb', 'OPEA': 'VLIp', 'OCDA': 'VLKA', 'OCDAE': 'VLKA', 'ONAE': 'VLL', 'ODNA': 'VLL', 'ODAlOV': 'VLpH', 'ODOV': 'VLpH', 'ODAlAn': 'VLtH', 'ODOA': 'VLtH', 'OCo': 'VlV', 'OEMTP': 'VMAP', 'OSMLP': 'VMAP', 'OAMAS': 'VMBM', 'OAMM': 'VMBM', 'ODPS': 'VMD', 'OMi': 'VMDP', 'OOPSM': 'VMDP', 'OACAMIL': 'VMILAK', 'OACMIL': 'VMILAK', 'OAMC': 'VmKI', 'OMob': 'VMob', 'OSM': 'VMS', 'ONM': 'VMSch', 'OCM': 'VMSV', 'OCSM': 'VMSV', 'OBLF': 'VMWG', 'OLAL': 'VMWG', 'OOPT': 'VNEF', 'OORT': 'VNEF', 'OCNE': 'VNEK', 'OCAl': 'VNem', 'OIAl': 'VNem', 'OUS': 'VNF', 'OAPub': 'VöB', 'OCOV': 'VOCV', 'OSO': 'VOD', 'OOBE': 'VOEW', 'OOSE': 'VOEW', 'OOSG': 'VOGW', 'OCoR': 'VORA', 'OCoR-DFI': 'VORA-EDI', 'OTN': 'VOSA', 'OPoA': 'VPA', 'OPPE': 'VPA', 'ORCPP': 'VPABP', 'OPPCPers': 'VPABP', 'OSAss': 'VPAV', 'RPAss': 'VPAV', 'OTV': 'VPB', 'OPIE': 'VPeA', 'OPAR': 'VPEV', 'ORPAR': 'VPEV', 'OAPA': 'VPGA', 'ORInt': 'VPiB', 'OMP-OFEV': 'VpM-BAFU', 'OMF-UFAM': 'VpM-BAFU', 'OMP-OFAG': 'VpM-BLW', 'OMF-UFAG': 'VpM-BLW', 'OOP EPF': 'VPO ETH', 'OOP-PF': 'VPO ETH', 'OOPC': 'VPOB', 'OFipo': 'VPofi', 'OFiPo': 'VPofi', 'OLOP': 'VPOG', 'OOP': 'VPOG', 'ODP': 'VPR', 'OMAP': 'VPRG', 'ODFCLSI': 'VPRG', 'OPOVA': 'VPRH', 'OAOVA': 'VPRH', 'OPPr': 'VPrP', 'OPPrim': 'VPrP', 'OPSP': 'VPS', 'OCSP': 'VPSP', 'OEnB': 'VPV', 'RPB': 'VPV', 'OPAPIF': 'VPVE', 'ORPM': 'VPVK', 'ORPMUE': 'VPVKEU', 'RP-EPF 1': 'VR-ETH 1', 'RP-PF 1': 'VR-ETH 1', 'RP-EPF 2': 'VR-ETH 2', 'RP-PF 2': 'VR-ETH 2', 'OCBD': 'VRA', 'OCQO': 'VRA', 'RPEC': 'VRAB', 'RPIC': 'VRAB', 'ORSAE': 'VREG', 'OSCR': 'VRKD', 'ORésDAlan': 'VRLtH', 'ORDOA': 'VRLtH', 'OPR': 'VRP', 'ORCPL': 'VRSL', 'OReq': 'VRSL', 'ORA': 'VRV-L', 'ONCA': 'VRV-L', 'OPCF': 'VSB', 'OEMCN': 'VSBN', 'OSMCN': 'VSBN', 'OAbCV': 'VSFK', 'ODCS': 'VSFS', 'ODSRS': 'VSFS', 'OOCCR-OFROU': 'VSKV-ASTRA', 'OOCCS-USTR': 'VSKV-ASTRA', 'OMSA': 'VSL', 'OSMP': 'VSMS', 'OMSM': 'VSMS', 'ODiTr': 'VSoTr', 'ODiT': 'VSoTr', 'OPPBE': 'VSPA', 'OPBE': 'VSPA', 'OPESp': 'VSpoFöP', 'OPPSpo': 'VSpoFöP', 'OPPB': 'VSPS', 'ORS': 'VSR', 'ORSA': 'VSRL', 'OSJo': 'VSS', 'OSG': 'VSS', 'OOST': 'VST', 'OOSI': 'VST', 'ORCS': 'VStFG', 'ORCel': 'VStFG', 'LIA': 'VStG', 'LIP': 'VStG', 'DPA': 'VStrR', 'OIA': 'VStV', 'OIPrev': 'VStV', 'OSAA': 'VSUV', 'OSAI': 'VSUV', 'OFDPP': 'VSVB', 'OFDP': 'VSVB', 'OEIT': 'VSZV', 'OIET': 'VSZV', 'OCVM': 'VTE', 'OVF': 'VTE', 'OAP': 'VTM', 'OAAP': 'VTM', 'OSPA': 'VTNP', 'OSOAn': 'VTNP', 'OETV': 'VTS', 'OPAnAb': 'VTSchS', 'OPAnMac': 'VTSchS', 'OPAT': 'VWS', 'OPrTec': 'VtVtH', 'OOr': 'VUB', 'OOr-DEFR': 'VUB-WBF', 'OAO-DEFR': 'VUB-WBF', 'OFSPers': 'VUFB', 'OCPCP': 'VUKBK', 'OCIPP': 'VUKBK', 'OACM': 'VUM', 'OAAM': 'VUM', 'OSCPT': 'VÜPF', 'OPA': 'VUV', 'OPI': 'VUV', 'OVid-TP': 'VüV-ÖV', 'OVsor-TP': 'VüV-ÖV', 'OROPD': 'VUZPE', 'OROPS': 'VUZPE', 'OAA': 'VVA', 'OAE': 'VVA', 'OPC': 'VVAG', 'ODiC': 'VVAG', 'OOLDI': 'VVAwG', 'O-ODI-DFAE': 'VVAwG', 'OISec': 'VVE', 'OFIM': 'VVE', 'OLED': 'VVEA', 'OPSR': 'VVEA', 'LCA': 'VVG', 'OTeA': 'VVK', 'OCA-DFI': 'VVK-EDI', 'OTeA-DFI': 'VVK-EDI', 'OMAH': 'VVMH', 'OMPAH': 'VVMH', 'OOUS': 'VVNF', 'OUUS': 'VVNF', 'OST-SCPT': 'VVS-ÜPF', 'OPSE': 'VVSG', 'OPreS': 'VVSG', 'OERE': 'VVWAL', 'OEAE': 'VVWAL', 'LALM': 'VWBG', 'LMAM': 'VWBG', 'OLCAP': 'VWEG', 'OFSI': 'VWEV', 'OCMSD': 'VWEV', 'OSS': 'VWL', 'OAEP': 'VWLV', 'OPATE': 'VWS', 'PA': 'VwVG', 'OASA': 'VZAE', 'LGEL': 'VZEG', 'LPIL': 'VZEG', 'OSCSE': 'VZertES', 'OFiEle': 'VZertES', 'ORFI': 'VZG', 'RFF': 'VZG', 'ONCAF': 'VZSchB', 'OAC': 'VZV', 'OASM': 'VZVM', 'OAVM': 'VZVM', 'LFo': 'WaG', 'OFo': 'WaV', 'LFCo': 'WeBiG', 'OFCo': 'WeBiV', 'OEPL': 'WEFV', 'OPPA': 'WEFV', 'LCAP': 'WEG', 'LArm': 'WG', 'LAMO': 'WHG', 'LTEO': 'WPEG', 'OTEO': 'WPEV', 'OIRH': 'WResV', 'OREI': 'WResV', 'LFH': 'WRG', 'LUFI': 'WRG', 'OFH': 'WRV', 'OUFI': 'WRV', 'LPAP': 'WSchG', 'LPSt': 'WSchG', 'OPAP': 'WSchV', 'OPSt': 'WSchV', 'OMT': 'WTO', 'OArm': 'WV', 'LUMMP': 'WZG', 'LUMP': 'WZG', 'RDE': 'WZV', 'ODA': 'WZV', 'OROEM': 'WZVV', 'ORUAM': 'WZVV', 'LUsC': 'ZAG', 'LCoe': 'ZAG', 'LFisE': 'ZBstG', 'LFR': 'ZBstG', 'LSC': 'ZDG', 'ODSC': 'ZDUeV', 'OSCi': 'ZDV', 'OSCi-DEFR': 'ZDV-WBF', 'LDIF': 'ZEBG', 'LSIF': 'ZEBG', 'LOC': 'ZentG', 'LUC': 'ZentG', 'SCSE': 'ZertES', 'FiEle': 'ZertES', 'Ltém': 'ZeugSG', 'LPTes': 'ZeugSG', 'OTém': 'ZeugSV', 'OPTes': 'ZeugSV', 'OADou': 'ZEV', 'OADo': 'ZEV', 'LD': 'ZG', 'CC': 'ZGB', 'LCPI': 'ZISG', 'OCMétr': 'ZMessV', 'OCMetr': 'ZMessV', 'CPC': 'ZPO', 'CCoop-ESF': 'ZSAV-BiZ', 'CColl-SFS': 'ZSAV-BiZ', 'CCoop-HE': 'ZSAV-HS', 'ConSU': 'ZSAV-HS', 'OAASF': 'ZSTEBV', 'OEEC': 'ZStGV', 'OESC': 'ZStGV', 'OEC': 'ZStV', 'OSC': 'ZStV', 'OPCi': 'ZSV', 'LTaD': 'ZTG', 'LTD': 'ZTG', 'LAS': 'ZUG', 'OComp-OSPro': 'ZustV-PrSV', 'OAdd': 'ZuV', 'OD': 'ZV', 'OD-OFDF': 'ZV-BAZG', 'OD-UDSC': 'ZV-BAZG', 'OD-DFF': 'ZV-EFD', 'OA-DFJP': 'ZV-EJPD', 'OA-DFGP': 'ZV-EJPD', 'LRS': 'ZWG', 'LASec': 'ZWG', 'ORSec': 'ZWV', 'OASec': 'ZWV'}
print(f'multilang entries: {len(MULTILANG_ABBR)}')

In [ ]:
# Cell 3 — citation parsing, normalization, corpus index, granularity filter
ART_PATTERN = re.compile(
    r'^Art\.\s*(?P<art>\d+[a-z]?(?:bis|ter|quater)?)'
    r'(?:\s*Abs\.\s*(?P<abs>\d+[a-z]?(?:bis|ter)?))?'
    r'(?:\s*lit\.\s*(?P<lit>[a-zA-Z]+))?'
    r'(?:\s*Ziff\.\s*(?P<ziff>\d+))?'
    r'\s+(?P<code>[A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9\.\-]+)$'
)

def parse_article(c):
    m = ART_PATTERN.match(c.strip()); return m.groupdict() if m else None

def normalize_whitespace(c):
    s = re.sub(r'\s+', ' ', c.strip())
    s = re.sub(r'\bArt\.(?=\d)', 'Art. ', s)
    s = re.sub(r'\bAbs\.(?=\d)', 'Abs. ', s)
    s = re.sub(r'\blit\.(?=[a-zA-Z])', 'lit. ', s)
    s = re.sub(r'\bZiff\.(?=\d)', 'Ziff. ', s)
    return s

def expand_multilang(c):
    out = [c]
    m = parse_article(c)
    if m and m['code'] in MULTILANG_ABBR:
        code_de = MULTILANG_ABBR[m['code']]
        out.append(c[:c.rfind(m['code'])] + code_de)
    return out

print('Loading laws & court citation sets ...')
laws_df = pd.read_csv(DATA / 'laws_de.csv')
laws_cits = set(laws_df['citation'].astype(str))
court_cits = set()
with open(DATA / 'court_considerations.csv', encoding='utf-8') as f:
    r = csv.reader(f); next(r, None)
    for row in r:
        if row: court_cits.add(row[0])
CORPUS = laws_cits | court_cits
print(f'corpus citations: laws={len(laws_cits):,}  court={len(court_cits):,}  combined={len(CORPUS):,}')

PARENT_TO_CHILDREN = defaultdict(list)
for c in laws_cits:
    m = ART_PATTERN.match(c)
    if not m: continue
    parent = f"Art. {m.group('art')} {m.group('code')}"
    if c != parent:
        PARENT_TO_CHILDREN[parent].append(c)
print(f'parents with children: {len(PARENT_TO_CHILDREN):,}')

def resolve_against_corpus(cit):
    c = normalize_whitespace(cit)
    if c in CORPUS: return [c]
    for v in expand_multilang(c):
        if v != c and v in CORPUS: return [v]
        if v != c and v in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[v])
    if c in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[c])
    m = parse_article(c)
    if m:
        parent = f"Art. {m['art']} {m['code']}"
        if parent != c and parent in CORPUS: return [parent]
        for v in expand_multilang(parent):
            if v != c and v in CORPUS: return [v]
            if v != c and v in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[v])
    return []

def granularity_filter(cits):
    cs = set(cits); keep=[]
    for c in cits:
        kids = PARENT_TO_CHILDREN.get(c)
        if kids and any(k in cs for k in kids): continue
        keep.append(c)
    return keep

def dedup(cits):
    seen=set(); out=[]
    for c in cits:
        if c not in seen: seen.add(c); out.append(c)
    return out

# sanity: train resolve rate
train_df = pd.read_csv(DATA / 'train.csv')
def split_cits(s):
    if not isinstance(s, str) or not s.strip(): return []
    return [t.strip() for t in s.split(';') if t.strip()]
total=resolved=0
for _, r in train_df.iterrows():
    for g in split_cits(r['gold_citations']):
        total+=1
        if resolve_against_corpus(g): resolved+=1
print(f'TRAIN gold resolve: {resolved}/{total} = {resolved/max(1,total):.4f}')
assert resolved/max(1,total) > 0.95, 'normalization regressed below 95% — investigate'


In [ ]:
# Cell 4 — BGE-M3 encoder(s): primary cuda:0, secondary cuda:1 (dual-GPU encoding)
import torch, gc, os
from pathlib import Path
from sentence_transformers import SentenceTransformer

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
ENCODER_DEVICE = 'cuda:0' if N_GPU >= 1 else 'cpu'

def discover_bge_m3_paths():
    paths = []
    ki = Path('/kaggle/input')
    if ki.exists():
        for root, dirs, files in os.walk(ki):
            if 'config.json' not in files: continue
            rl = root.lower()
            if 'bge' in rl or '/m3/' in rl + '/' or rl.endswith('/m3'):
                paths.append(root)
    paths.extend([
        '/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3',
        '/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1',
        '/kaggle/input/bge-m3', 'BAAI/bge-m3',
    ])
    seen=set(); out=[]
    for p in paths:
        if p not in seen: seen.add(p); out.append(p)
    return out

BGE_PATHS = discover_bge_m3_paths()
print('BGE-M3 candidates:', [p for p in BGE_PATHS[:4]])

def load_bge(device):
    for p in BGE_PATHS:
        try:
            m = SentenceTransformer(p, device=device)
            try: m.half()
            except Exception: pass
            try: m.max_seq_length = 512
            except Exception: pass
            print(f'  \u2705 BGE-M3 on {device} from {p}')
            return m
        except Exception as e:
            print(f'  {device} {p} failed: {type(e).__name__}: {str(e)[:70]}')
    return None

ENCODER = load_bge(ENCODER_DEVICE)
if ENCODER is None:
    raise RuntimeError('BGE-M3 not available. Attach model or Internet ON.')

# Secondary encoder on cuda:1 for dual-GPU bulk encoding (laws). Freed before LLM load.
ENCODER2 = None
if N_GPU >= 2:
    ENCODER2 = load_bge('cuda:1')

def encode(texts, batch_size=32, show_progress=False):
    return ENCODER.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                          show_progress_bar=show_progress, convert_to_numpy=True).astype('float32')
def encode_query(q):
    return encode([q], batch_size=1)

if torch.cuda.is_available():
    for i in range(N_GPU):
        free, total = torch.cuda.mem_get_info(i)
        print(f'  cuda:{i}  free={free/1e9:.1f}GB / total={total/1e9:.1f}GB')


In [ ]:
# Cell 5 — laws_de BM25 + BGE-M3 dense (DUAL-GPU parallel encode, cached)
import faiss
from rank_bm25 import BM25Okapi
import threading

TOKEN_RE = re.compile(r'[A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9]+')
def tokenize(text):
    if not isinstance(text, str): return []
    return [t.lower() for t in TOKEN_RE.findall(text)]

laws_citations = laws_df['citation'].astype(str).tolist()
laws_texts = laws_df['text'].fillna('').astype(str).tolist()

bm25_path = CACHE / 'laws_bm25.pkl'
if bm25_path.exists():
    print('Loading cached laws BM25 ...')
    with open(bm25_path, 'rb') as f: bm25 = pickle.load(f)
else:
    print('Building BM25 ...')
    t0=time.time(); bm25 = BM25Okapi([tokenize(t) for t in laws_texts])
    with open(bm25_path,'wb') as f: pickle.dump(bm25,f)
    print(f'  BM25 in {time.time()-t0:.1f}s')

faiss_path = CACHE / 'laws_dense.faiss'
if faiss_path.exists():
    print('Loading cached laws FAISS ...')
    laws_dense = faiss.read_index(str(faiss_path))
else:
    n = len(laws_texts)
    t0 = time.time()
    if ENCODER2 is not None:
        # Split in half, encode each half on its own GPU concurrently (threads — CUDA frees GIL)
        mid = n // 2
        halves = [(ENCODER, laws_texts[:mid], 'cuda:0'), (ENCODER2, laws_texts[mid:], 'cuda:1')]
        results = [None, None]
        def _enc(idx, model, texts, tag):
            print(f'  [{tag}] encoding {len(texts):,} ...', flush=True)
            results[idx] = model.encode(texts, batch_size=32, normalize_embeddings=True,
                                        show_progress_bar=False, convert_to_numpy=True).astype('float32')
            print(f'  [{tag}] done', flush=True)
        threads = [threading.Thread(target=_enc, args=(i, m, t, tag)) for i,(m,t,tag) in enumerate(halves)]
        for th in threads: th.start()
        for th in threads: th.join()
        emb = np.vstack([results[0], results[1]])
        print(f'  DUAL-GPU encode done in {time.time()-t0:.1f}s  shape={emb.shape}')
    else:
        # single-GPU chunked fallback
        parts=[]
        CHUNK=10000
        for s in range(0,n,CHUNK):
            e=min(s+CHUNK,n)
            parts.append(ENCODER.encode(laws_texts[s:e], batch_size=32, normalize_embeddings=True,
                                        show_progress_bar=False, convert_to_numpy=True).astype('float32'))
            print(f'  [{e}/{n}] {time.time()-t0:.1f}s', flush=True)
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        emb=np.vstack(parts)
        print(f'  single-GPU encode done in {time.time()-t0:.1f}s  shape={emb.shape}')

    laws_dense = faiss.IndexFlatIP(emb.shape[1]); laws_dense.add(emb)
    faiss.write_index(laws_dense, str(faiss_path))
    del emb; gc.collect()

# Free the 2nd encoder now — its VRAM goes to the LLM (device_map='auto' uses both GPUs)
if ENCODER2 is not None:
    del ENCODER2; ENCODER2 = None
    import time as _t
    for _ in range(3):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.synchronize()
        _t.sleep(2)
    print('  freed cuda:1 BGE-M3 (reserved for LLM)')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            print(f'    cuda:{i} free={free/1e9:.1f}GB')

print(f'laws FAISS ready: ntotal={laws_dense.ntotal}  d={laws_dense.d}')


In [ ]:
# Cell 6 — BGE-only court 2-stage indexing (leading cases, ~14 min)
# Data: BGE = 96K E-rows / ~7K judgments (3.9% of court), but 2x BGer in val gold.
# Skip the 1.95M BGer individual rulings (78% of cost, 13% of gold).
import faiss
RE_BGE_ONLY = re.compile(r'^(BGE\s+\d+\s+[IVX]+\s+\d+)')

court_meta_path = CACHE / 'bge_court_meta.pkl'
court_jfaiss = CACHE / 'bge_court_j.faiss'
court_efaiss = CACHE / 'bge_court_e.faiss'
E_TRUNC = 400

if USE_COURT and court_meta_path.exists() and court_jfaiss.exists() and court_efaiss.exists():
    print('Loading cached BGE court indexes ...')
    court_meta = pickle.load(open(court_meta_path,'rb'))
    j_dense = faiss.read_index(str(court_jfaiss))
    e_dense = faiss.read_index(str(court_efaiss))
elif USE_COURT:
    print('Streaming court_considerations.csv, keeping BGE rows only ...')
    bucket=defaultdict(list); e_cits=[]; e_jids=[]; e_texts=[]
    t0=time.time(); E_CAP, CHAR_CAP=30,4000
    with open(DATA/'court_considerations.csv',encoding='utf-8') as f:
        r=csv.reader(f); next(r,None)
        for row in r:
            if not row or len(row)<2: continue
            cit,text=row[0],row[1] or ''
            m=RE_BGE_ONLY.match(cit)
            if not m: continue              # BGE only
            jid=m.group(1)
            e_cits.append(cit); e_jids.append(jid); e_texts.append(text)
            if len(bucket[jid])<E_CAP: bucket[jid].append(text)
    print(f'  BGE rows={len(e_cits):,}  judgments={len(bucket):,}  in {time.time()-t0:.1f}s')

    judgment_ids=list(bucket.keys())
    j2idx={j:i for i,j in enumerate(judgment_ids)}
    judgment_texts=['\n'.join(bucket[j])[:CHAR_CAP] for j in judgment_ids]
    e_to_jidx=[j2idx[j] for j in e_jids]

    print(f'Encoding {len(judgment_texts):,} BGE judgments + {len(e_texts):,} E-rows on {ENCODER_DEVICE} ...')
    t0=time.time()
    j_emb=ENCODER.encode(judgment_texts,batch_size=32,normalize_embeddings=True,
                         show_progress_bar=False,convert_to_numpy=True).astype('float32')
    j_dense=faiss.IndexFlatIP(j_emb.shape[1]); j_dense.add(j_emb); del j_emb; gc.collect()
    e_emb=ENCODER.encode(e_texts,batch_size=32,normalize_embeddings=True,
                         show_progress_bar=True,convert_to_numpy=True).astype('float32')
    e_dense=faiss.IndexFlatIP(e_emb.shape[1]); e_dense.add(e_emb)
    print(f'  encoded in {time.time()-t0:.1f}s')
    e_texts_trunc=[t[:E_TRUNC] for t in e_texts]
    del e_emb,e_texts; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    court_meta={'judgment_ids':judgment_ids,'e_citations':e_cits,
                'e_to_jidx':e_to_jidx,'e_texts_trunc':e_texts_trunc}
    pickle.dump(court_meta,open(court_meta_path,'wb'))
    faiss.write_index(j_dense,str(court_jfaiss))
    faiss.write_index(e_dense,str(court_efaiss))
else:
    court_meta={'judgment_ids':[],'e_citations':[],'e_to_jidx':[],'e_texts_trunc':[]}
    j_dense=None; e_dense=None

if USE_COURT:
    print(f'BGE court ready: judgments={len(court_meta["judgment_ids"]):,}  E-rows={len(court_meta["e_citations"]):,}')


In [ ]:
# Cell 7 — NLLB EN→DE translation on cuda:1 (fail-soft)
# Put NLLB on cuda:0 (with BGE-M3) so cuda:1 stays free for the LLM
SECOND_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

TRANSLATOR = None
TRANSLATOR_TOK = None
if USE_TRANSLATE:
    NLLB_PATHS = [
        '/kaggle/input/nllb-200-distilled/nllb-200-distilled-600M',
        '/kaggle/input/nllb-200-distilled-600m',
        '/kaggle/input/nllb-200-distilled',
        'facebook/nllb-200-distilled-600M',
    ]
    # also discover from /kaggle/input/models/
    extra = []
    ki = Path('/kaggle/input')
    if ki.exists():
        for root, dirs, files in os.walk(ki):
            if 'config.json' in files and 'nllb' in root.lower():
                extra.append(root)
    NLLB_PATHS = extra + NLLB_PATHS

    for p in NLLB_PATHS:
        try:
            from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
            TRANSLATOR_TOK = AutoTokenizer.from_pretrained(p, src_lang='eng_Latn')
            TRANSLATOR = AutoModelForSeq2SeqLM.from_pretrained(p, torch_dtype=torch.float16)
            TRANSLATOR = TRANSLATOR.to(SECOND_DEVICE)
            print(f'NLLB loaded on {SECOND_DEVICE} from {p}')
            break
        except Exception as e:
            print(f'  NLLB {p} failed: {type(e).__name__}: {str(e)[:80]}')
    if TRANSLATOR is None:
        print('  NLLB unavailable — fallback identity translate')

def translate_to_de(text):
    if TRANSLATOR is None: return text
    try:
        inputs = TRANSLATOR_TOK(text, return_tensors='pt', truncation=True, max_length=512)
        dev = next(TRANSLATOR.parameters()).device
        inputs = {k: v.to(dev) for k, v in inputs.items()}
        forced = TRANSLATOR_TOK.convert_tokens_to_ids('deu_Latn')
        with torch.no_grad():
            out = TRANSLATOR.generate(**inputs, forced_bos_token_id=forced,
                                       max_new_tokens=384, num_beams=1)
        return TRANSLATOR_TOK.batch_decode(out, skip_special_tokens=True)[0]
    except Exception:
        return text

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'  cuda:{i}  free={free/1e9:.1f}GB / total={total/1e9:.1f}GB')


In [ ]:
# Cell 9 — universal defaults + helpers (topic anchors NOT force-injected this time)
# Lesson from phase 6: hand-crafted topic anchors overfit val & crowded out signal.
# Here we keep only the universal default (Art. 100 Abs. 1 BGG: 9/10 val) and let the
# LLM generate the rest. classify_topic kept only for optional light prior.

UNIVERSAL_DEFAULTS = ['Art. 100 Abs. 1 BGG']

def get_universal():
    out = []
    for d in UNIVERSAL_DEFAULTS:
        r = resolve_against_corpus(d)
        if r: out.append(r[0])
    return dedup(out)

# Citation-form helpers for quality control
import re as _re_q
_SR_NUMERIC_RE = _re_q.compile(r'\s(\d{3}(?:\.\d+)+(?:[a-z])?)\s*$')
def code_of(cit):
    parts = cit.rsplit(' ', 1)
    return parts[-1] if parts else cit
def is_sr_numeric(cit):
    return bool(_SR_NUMERIC_RE.search(cit))


In [ ]:
# Cell 9 — LLM loader: Mistral-7B-Instruct (4-bit, SINGLE GPU, OOM-proof)
# Root-cause fixes vs prior failures:
#  1. 7B 4-bit ~5GB loads entirely on ONE GPU (cuda:1) — no device_map spreading, no max_memory
#  2. cleanup (del+empty_cache) after each failed attempt — prevents cascade OOM
#  3. only searches mistral-7b paths — ignores broken Nemo / 24B even if still attached
import torch, gc, os
from pathlib import Path

LLM = None; LLM_TOK = None; LLM_FAILED = False
def pick_freest_gpu():
    if not torch.cuda.is_available():
        return 'cpu'
    best, best_free = 0, -1
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'  cuda:{i} free={free/1e9:.1f}GB')
        if free > best_free:
            best, best_free = i, free
    print(f'  -> picking cuda:{best} (most free)')
    return f'cuda:{best}'

import time as _t
# settle memory before reading free amounts (prior cells may leave transient allocations)
for _ in range(3):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()
    _t.sleep(2)
LLM_DEVICE = pick_freest_gpu()

def discover_llm():
    cands = []
    ki = Path('/kaggle/input')
    if ki.exists():
        for root, dirs, files in os.walk(ki):
            if 'config.json' in files:
                rl = root.lower()
                if 'mistral-7b' in rl or 'mistral_7b' in rl or ('mistral' in rl and '7b' in rl):
                    cands.append(root)
    cands += [
        '/kaggle/input/models/mistral-ai/mistral/pytorch/7b-instruct-v0.1-hf/1',
        'mistralai/Mistral-7B-Instruct-v0.1',
        'mistralai/Mistral-7B-Instruct-v0.2',
    ]
    seen=set(); out=[]
    for c in cands:
        if c not in seen: seen.add(c); out.append(c)
    return out

bnb_cfg = None
try:
    import bitsandbytes  # noqa
    from transformers import BitsAndBytesConfig
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
    print('4-bit config ready')
except ImportError:
    print('no bitsandbytes — fp16 (7B fp16 ~14GB still fits one T4)')

print('LLM candidates:')
for p in discover_llm():
    print('  ', p)

from transformers import AutoModelForCausalLM, AutoTokenizer
for p in discover_llm():
    try:
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        try:
            LLM_TOK = AutoTokenizer.from_pretrained(p)
        except Exception:
            LLM_TOK = AutoTokenizer.from_pretrained(p, use_fast=False)
        kw = {'device_map': {'': LLM_DEVICE}}   # explicit: load fully on the freest GPU
        if bnb_cfg is not None: kw['quantization_config'] = bnb_cfg
        else: kw['torch_dtype'] = torch.float16
        LLM = AutoModelForCausalLM.from_pretrained(p, **kw)
        LLM.eval()
        print(f'\u2705 LLM loaded from {p} on {LLM_DEVICE} ({"4-bit" if bnb_cfg else "fp16"})')
        break
    except Exception as e:
        print(f'  failed {p}: {type(e).__name__}: {str(e)[:120]}')
        try: del LLM
        except Exception: pass
        LLM = None
        for _ in range(3):
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache(); torch.cuda.synchronize()
            _t.sleep(2)
        # re-pick freest GPU after cleanup (memory landscape may have changed)
        LLM_DEVICE = pick_freest_gpu()

if LLM is None:
    LLM_FAILED = True
    print('  No LLM loaded — fallback to retrieval-only')

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'  cuda:{i} free={free/1e9:.1f}GB / {total/1e9:.1f}GB')

def llm_generate(prompt, max_new_tokens=400):
    if LLM is None or LLM_FAILED:
        return ''
    try:
        try:
            dev = LLM.get_input_embeddings().weight.device
        except Exception:
            dev = LLM_DEVICE
        if hasattr(LLM_TOK, 'apply_chat_template') and getattr(LLM_TOK, 'chat_template', None):
            enc = LLM_TOK.apply_chat_template([{'role':'user','content':prompt}],
                                              return_tensors='pt', add_generation_prompt=True,
                                              return_dict=True)
        else:
            enc = LLM_TOK(prompt, return_tensors='pt', truncation=True, max_length=4096)
        enc = {k: v.to(dev) for k, v in enc.items()}
        in_len = enc['input_ids'].shape[1]
        with torch.no_grad():
            out = LLM.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                               repetition_penalty=1.3, no_repeat_ngram_size=4,
                               pad_token_id=(LLM_TOK.eos_token_id or 0))
        return LLM_TOK.decode(out[0][in_len:], skip_special_tokens=True)
    except Exception as e:
        import traceback
        print(f'  llm_generate exc: {type(e).__name__}: {e}')
        traceback.print_exc()
        return ''


In [ ]:
# Cell 10 — topic taxonomy + LLM classifier + topic-driven retrieval + predict
"""Phase 10 — closed-set topic taxonomy for LLM classification → topic-driven retrieval.

Design (per user's direction):
  - The LLM is shown this FIXED list (name + description) and outputs only the IDs it picks.
    It never invents a topic and never writes a citation → no hallucination.
  - Each topic maps deterministically to a German search query (query_de) + priority law
    codes (codes). The picked topics drive multi-query BGE-M3 retrieval over laws + court.

Derived from val gold law-code distribution (measured):
  BGE 69, ZGB 39, StPO 36, BGer 33, OR 18, BGG 13, StGB 12, IVG 10, ATSG 7, StBOG 4,
  ZPO 4, BV 3, IPRG 2, SchKG 1.
  → BGE+BGer (case law) = 41% of gold; BGG (federal appeal) is near-universal.
"""

# id → (name, description shown to LLM, German search query, priority law codes)
TOPICS = {
    1:  ("detention_stpo",
         "Pre-trial / security detention, coercive measures in criminal procedure "
         "(grounds for detention, flight/collusion/recidivism risk, detention appeals).",
         "Untersuchungshaft Sicherheitshaft Haftgründe Fluchtgefahr Kollusionsgefahr "
         "Wiederholungsgefahr Haftbeschwerde Haftentlassung Verhältnismässigkeit",
         ["StPO", "BV", "StBOG"]),

    2:  ("criminal_procedure",
         "Criminal procedure generally (charges, evidence, appeals/Beschwerde in criminal "
         "matters, costs, defence rights) — when the dispute is procedural, not the offence itself.",
         "Strafverfahren Strafprozess Beschwerde Berufung Beweis Verfahrenskosten "
         "amtliche Verteidigung Rechtsmittel",
         ["StPO", "StBOG", "BV"]),

    3:  ("criminal_substantive",
         "Substantive criminal law — a specific offence (theft, fraud, assault, "
         "disloyal management/embezzlement, forgery, etc.) and its elements/penalty.",
         "Straftat Tatbestand Diebstahl Betrug Veruntreuung ungetreue Geschäftsbesorgung "
         "Urkundenfälschung Körperverletzung Strafzumessung Vorsatz",
         ["StGB"]),

    4:  ("family_zgb",
         "Family law — divorce, marriage, spousal/child maintenance, parental "
         "responsibility/custody, child protection measures.",
         "Scheidung Ehe Unterhalt Kindesunterhalt elterliche Sorge Obhut "
         "Kindesschutz Beistandschaft Besuchsrecht",
         ["ZGB"]),

    5:  ("inheritance_zgb",
         "Inheritance / succession — wills, intestate succession, compulsory portion, "
         "disinheritance, inheritance contracts, executors.",
         "Erbrecht Erbschaft Testament letztwillige Verfügung gesetzliche Erbfolge "
         "Pflichtteil Enterbung Erbvertrag Willensvollstrecker",
         ["ZGB"]),

    6:  ("property_zgb",
         "Property / rights in rem — ownership, possession, land register, easements, "
         "mortgages, co-ownership, acquisition in good faith.",
         "Eigentum Besitz Grundbuch Grunddienstbarkeit Grundpfand Miteigentum "
         "Stockwerkeigentum gutgläubiger Erwerb Sachenrecht",
         ["ZGB"]),

    7:  ("tenancy_or",
         "Lease / tenancy law — residential or commercial lease, rent, defects, "
         "termination, protection against unfair termination.",
         "Mietvertrag Miete Mietzins Mängel Kündigung ausserordentliche Kündigung "
         "Erstreckung Anfechtung Kündigungsschutz",
         ["OR"]),

    8:  ("employment_or",
         "Employment law — employment contract, wages, termination/dismissal, "
         "wrongful termination, protection of the employee.",
         "Arbeitsvertrag Lohn Kündigung missbräuchliche Kündigung fristlose Entlassung "
         "Arbeitszeugnis Konkurrenzverbot",
         ["OR"]),

    9:  ("contract_or",
         "General contract law and specific contracts other than lease/employment — "
         "sale, mandate/agency, contract for work (Werkvertrag), formation, performance, breach.",
         "Vertrag Kaufvertrag Werkvertrag Auftrag Mängel Gewährleistung Verzug "
         "Vertragsverletzung Willensmängel Verjährung",
         ["OR"]),

    10: ("tort_liability_or",
         "Tort / non-contractual liability and damages — fault, causation, damage, "
         "strict liability, product/owner liability.",
         "unerlaubte Handlung Haftung Schadenersatz Verschulden Kausalzusammenhang "
         "Genugtuung Kausalhaftung Werkeigentümerhaftung",
         ["OR"]),

    11: ("debt_enforcement_schkg",
         "Debt enforcement and bankruptcy — Betreibung, attachment, opposition, "
         "Rechtsöffnung, bankruptcy, composition.",
         "Betreibung Schuldbetreibung Konkurs Pfändung Arrest Rechtsöffnung "
         "Rechtsvorschlag Kollokation",
         ["SchKG"]),

    12: ("social_insurance",
         "Social insurance — disability (IV), old-age (AHV), accident (UV), health (KV) "
         "insurance; benefits, incapacity, the general social-insurance procedure (ATSG).",
         "Invalidenversicherung IV-Rente Erwerbsunfähigkeit Arbeitsunfähigkeit "
         "Sozialversicherung ATSG Unfallversicherung Altersrente",
         ["IVG", "ATSG", "UVG", "AHVG", "KVG"]),

    13: ("tax",
         "Tax law — direct federal/cantonal tax, VAT, withholding tax, assessment, deductions.",
         "Steuer direkte Bundessteuer Mehrwertsteuer Verrechnungssteuer "
         "Veranlagung Steuerabzug steuerbares Einkommen",
         ["DBG", "MWSTG", "StHG", "VStG"]),

    14: ("commercial_banking",
         "Commercial, banking and financial-market law, company law — companies, "
         "directors' liability, banking, securities, financial-market supervision; "
         "often with private-international-law (cross-border) elements.",
         "Gesellschaftsrecht Aktiengesellschaft Verwaltungsrat Bank Bankvertrag "
         "Finanzmarkt Sorgfaltspflicht Geldwäscherei internationales Privatrecht",
         ["OR", "BankG", "FINMAG", "IPRG", "GwG"]),

    15: ("civil_procedure",
         "Civil procedure — jurisdiction, lis pendens, evidence, provisional measures, "
         "appeals in civil matters (when the dispute is procedural).",
         "Zivilprozess Zuständigkeit Rechtshängigkeit Beweis vorsorgliche Massnahmen "
         "Berufung Beschwerde ZPO Klage",
         ["ZPO", "IPRG"]),

    16: ("administrative_public",
         "Administrative / public law — migration/asylum, building & planning, "
         "data protection, public-law disputes and their procedure.",
         "Verwaltungsrecht öffentliches Recht Migration Asyl Aufenthalt Datenschutz "
         "Verfügung Verwaltungsbeschwerde",
         ["AIG", "DSG", "RPG", "VwVG"]),

    17: ("federal_appeal",
         "ALWAYS APPLICABLE when the case reaches the Federal Supreme Court "
         "(Bundesgericht) on appeal — the procedural framework of the appeal itself "
         "(admissibility, deadline, grounds, cognition). Pick this for nearly every case.",
         "Beschwerde ans Bundesgericht Beschwerdefrist Beschwerdelegitimation "
         "Rügeprinzip Kognition Eintreten Endentscheid",
         ["BGG"]),

    18: ("constitutional_rights",
         "Fundamental / constitutional rights invoked in the dispute — right to be heard, "
         "personal liberty, equality, proportionality, fair trial (also ECHR/EMRK).",
         "Grundrecht rechtliches Gehör persönliche Freiheit Rechtsgleichheit "
         "Verhältnismässigkeit faires Verfahren EMRK",
         ["BV", "EMRK"]),
}

# Case-law (BGE/BGer) is 41% of gold and is retrieved from the court index using the
# same query_de keywords of the picked topics (no separate topic needed — court search
# runs on the union of picked-topic queries).


RE_ART_Q  = re.compile(r'\bArt(?:icle)?\.?\s*(\d+[a-z]?(?:bis|ter|quater)?)'
                       r'(?:\s*(?:Abs\.|al\.|para\.)\s*(\d+))?'
                       r'(?:\s*(?:lit\.|let\.)\s*([a-z]))?'
                       r'\s+([A-Z][A-Za-zÄÖÜäöüß0-9\.\-]+)')
RE_BGE_Q  = re.compile(r'\bBGE\s+(\d+)\s+([IVX]+)\s+(\d+)(?:\s+E\.?\s*([\d\.]+))?')
RE_BGER_Q = re.compile(r'\b(\d+[A-Za-z]_\d+/\d+)(?:\s+E\.?\s*([\d\.]+))?')

def extract_seeds(query):
    seeds=[]
    for m in RE_ART_Q.finditer(query):
        art, abs_, lit, c = m.groups()
        parts=[f'Art. {art}']
        if abs_: parts.append(f'Abs. {abs_}')
        if lit:  parts.append(f'lit. {lit}')
        parts.append(c)
        seeds.extend(resolve_against_corpus(' '.join(parts)))
    for m in RE_BGE_Q.finditer(query):
        vol,book,page,e=m.groups(); base=f'BGE {vol} {book} {page}'
        if e: seeds.extend(resolve_against_corpus(f'{base} E. {e}'))
        seeds.extend(resolve_against_corpus(base))
    for m in RE_BGER_Q.finditer(query):
        case,e=m.groups()
        if e: seeds.extend(resolve_against_corpus(f'{case} E. {e}'))
        seeds.extend(resolve_against_corpus(case))
    return dedup(seeds)

def reciprocal_rank_fusion(rankings, weights=None, k=60):
    if weights is None: weights=[1.0]*len(rankings)
    sc={}
    for rk,w in zip(rankings,weights):
        for r,cit in enumerate(rk): sc[cit]=sc.get(cit,0.0)+w/(k+r+1)
    return sc

def laws_search_one(q, top_k=100):
    bm=bm25.get_scores(tokenize(q)); bt=np.argsort(bm)[::-1][:300]
    rankings=[[laws_citations[i] for i in bt]]; weights=[0.3]
    qe=encode_query(q); _,I=laws_dense.search(qe,300)
    rankings.append([laws_citations[i] for i in I[0]]); weights.append(0.7)
    fused=reciprocal_rank_fusion(rankings,weights=weights)
    return [c for c,_ in sorted(fused.items(),key=lambda kv:-kv[1])[:top_k]]

E_EMB_RECON=None
if USE_COURT and e_dense is not None and hasattr(e_dense,'reconstruct_n'):
    try:
        print('Reconstructing BGE E-level embeddings ...')
        t0=time.time(); E_EMB_RECON=e_dense.reconstruct_n(0,e_dense.ntotal)
        print(f'  {E_EMB_RECON.shape} in {time.time()-t0:.1f}s')
    except Exception as exc: print('  reconstruct failed:',exc)

def court_search_one(q, top_k_j=40, top_k_e=20):
    if not USE_COURT or j_dense is None or e_dense is None or E_EMB_RECON is None: return []
    qe=encode_query(q); _,I=j_dense.search(qe,top_k_j)
    sel=set(int(i) for i in I[0] if i>=0)
    e2j=court_meta['e_to_jidx']
    eidx=[i for i,j in enumerate(e2j) if j in sel]
    if not eidx: return []
    es=E_EMB_RECON[eidx]; s=es@qe[0]
    order=np.argsort(-s)[:top_k_e]; ec=court_meta['e_citations']
    return [ec[eidx[k]] for k in order]

text_by_cit=dict(zip(laws_citations,laws_texts))
if USE_COURT and court_meta.get('e_texts_trunc'):
    for cit,txt in zip(court_meta['e_citations'],court_meta['e_texts_trunc']):
        if cit not in text_by_cit: text_by_cit[cit]=txt

import re as _q
_SR=_q.compile(r'\s(\d{3}(?:\.\d+)+(?:[a-z])?)\s*$')
def is_sr_numeric(c): return bool(_SR.search(c))
def code_of(c):
    if c.startswith('BGE ') or '_' in c.split(' ')[-1]: return None
    p=c.rsplit(' ',1); return p[-1] if len(p)==2 else None

def build_topic_prompt(query):
    lines=["You are a Swiss legal expert. Classify the CASE into legal areas by choosing "
           "from the FIXED list below. Output ONLY the matching numbers, comma-separated "
           "(e.g. \"1, 17, 18\"). Do NOT write any citation or explanation. "
           "Almost every case reaching the Federal Supreme Court includes 17, most include 18.\n"]
    for tid,(name,desc,_,_) in TOPICS.items():
        lines.append(f"{tid}. {name}: {desc}")
    lines.append(f"\nCASE:\n{query[:2500]}\n\nMatching numbers:")
    return "\n".join(lines)

def classify_topics(query):
    if LLM is None or LLM_FAILED:
        return [17, 18]
    out = llm_generate(build_topic_prompt(query), max_new_tokens=40)
    ids=set()
    for m in re.findall(r'\d+', out):
        v=int(m)
        if v in TOPICS: ids.add(v)
    if not ids: ids={17,18}
    ids.add(17)
    return sorted(ids)

def predict(query, qid=None, return_debug=False):
    seeds = extract_seeds(query)
    universal = get_universal()
    topic_ids = classify_topics(query)

    query_de = translate_to_de(query) if USE_TRANSLATE else query
    queries = [query, query_de] if query_de != query else [query]
    boost_codes = set()
    for tid in topic_ids:
        name, desc, qde, codes = TOPICS[tid]
        queries.append(qde); boost_codes.update(codes)

    laws_rankings = [laws_search_one(q, top_k=100) for q in queries]
    court_rankings = [court_search_one(q, top_k_e=20) for q in queries]
    laws_fused = reciprocal_rank_fusion(laws_rankings)
    court_fused = reciprocal_rank_fusion(court_rankings) if any(court_rankings) else {}

    def final_score(cit, base):
        co = code_of(cit)
        return base * (1.6 if (co and co in boost_codes) else 1.0)

    laws_ranked = sorted(((c, final_score(c, sc)) for c, sc in laws_fused.items()), key=lambda kv: -kv[1])
    court_ranked = sorted(court_fused.items(), key=lambda kv: -kv[1])
    laws_cits = [c for c, _ in laws_ranked if not is_sr_numeric(c)]
    court_cits = [c for c, _ in court_ranked]

    # PRECISION-FIRST compose (LB lesson: fewer, higher-confidence beats many):
    #   front = seeds (literal) + universal default (9/10 val)
    #   + top topic-boosted laws (main body)
    #   + only top 3 court BGE (court retrieval is noisy: ~7 FP per query otherwise)
    EMIT_CAP = 16
    COURT_KEEP = 3
    front = dedup(seeds + universal); fset = set(front)
    laws_body = [c for c in laws_cits if c not in fset][:EMIT_CAP]
    court_body = [c for c in court_cits if c not in fset and c not in set(laws_body)][:COURT_KEEP]
    out = granularity_filter(dedup(front + laws_body + court_body))[:EMIT_CAP]

    if return_debug:
        nbge=sum(1 for c in out if c.startswith('BGE '))
        return out, {'topics':[TOPICS[t][0] for t in topic_ids],'n_topics':len(topic_ids),
                     'n_seeds':len(seeds),'n_emit':len(out),'n_bge':nbge}
    return out


In [ ]:
# Cell 11 — Val Macro F1 + topic diagnostic
def per_query_f1(g,p):
    g,p=set(g),set(p)
    if not g and not p: return 1.0
    if not g or not p: return 0.0
    tp=len(g&p)
    if tp==0: return 0.0
    pr=tp/len(p); rc=tp/len(g); return 2*pr*rc/(pr+rc)
def macro_f1(gd,pd_):
    qs=set(gd)|set(pd_)
    return sum(per_query_f1(gd.get(q,[]),pd_.get(q,[])) for q in qs)/max(1,len(qs))

val=pd.read_csv(DATA/'val.csv')
gold={r['query_id']:split_cits(r['gold_citations']) for _,r in val.iterrows()}
print('Predicting val ...'); t0=time.time()
results={r['query_id']:predict(r['query'],r['query_id'],return_debug=True) for _,r in val.iterrows()}
print(f'  done in {time.time()-t0:.1f}s')
preds={q:p for q,(p,_) in results.items()}
f1=macro_f1(gold,preds)
print(f'\nVAL Macro F1 = {f1:.4f}\n')
print(f"{'qid':8s} {'F1':>6s} {'gld':>4s} {'emt':>4s} {'tp':>3s} {'bge':>3s}  topics")
for qid in sorted(gold):
    g=gold[qid]; pred,dbg=results[qid]
    tp=len(set(g)&set(pred))
    print(f"{qid:8s} {per_query_f1(g,pred):6.3f} {len(g):>4d} {len(pred):>4d} {tp:>3d} {dbg['n_bge']:>3d}  {','.join(dbg['topics'])}")


In [ ]:
# Cell 14 — submission.csv
test=pd.read_csv(DATA/'test.csv')
rows=[]
print(f'Predicting test ({len(test)} rows) ...')
t0=time.time()
for i,(_,r) in enumerate(test.iterrows(),1):
    cits=predict(r['query'], r['query_id'])
    rows.append({'query_id':r['query_id'],'predicted_citations':';'.join(cits)})
    if i%5==0: print(f'  {i}/{len(test)}  elapsed {time.time()-t0:.1f}s', flush=True)
print(f'  done in {time.time()-t0:.1f}s')

sub=pd.DataFrame(rows)
out=WORK/'submission.csv'
sub.to_csv(out,index=False)
print(f'\nWrote {out}  rows={len(sub)}')
print(sub.head(3).to_string(index=False))
counts=sub['predicted_citations'].str.count(';').add(1)
print(f'\nemit count  min={counts.min()}  mean={counts.mean():.1f}  max={counts.max()}')
allp=[]
for cs in sub['predicted_citations']: allp.extend(cs.split(';'))
uniq=set(allp)
inc=sum(1 for c in uniq if c in CORPUS)
print(f'corpus coverage: {inc}/{len(uniq)} = {inc/max(1,len(uniq)):.4f}')
